# Mission 3 ver25 — 4,000개 예비 학습 후 결과 저장·종료

Colab GPU 런타임에서 **런타임 → 모두 실행**을 누르세요.
Drive 연결 시 입력 ZIP이 있는 계정을 선택하세요. 별도 코드 업로드 없이 이 노트북과
기존 `mission3_ver5_inputs.zip`으로 실행합니다. 공개 사전학습 가중치는 실행 중 내려받습니다.
준비 셀이 실패하거나 런타임을 초기화했다면 마지막 셀만 실행하지 말고 처음부터 실행하세요.

**Qwen3-4B-Instruct-2507의 내부 LoRA adapter와 9개 증상 분류층**을 실제 Training 정답으로 학습합니다.
새로 학습한 KLUE 대조군, Qwen 단독, 고정 1:1 확률 평균을 같은 분할에서 비교합니다.
자유 문장 생성이나 외부 LLM 정답 생성은 하지 않습니다. 외부 학습 데이터도 추가하지 않습니다.
기본 실행은 Training fit 분할에서 중복 그룹과 증상 비율을 고려한 **약 4,000개로 두 모델을 1 epoch 학습**합니다.
중복 그룹을 나눌 수 없는 경우 실제 개수가 조금 달라질 수 있으며 선택 내역을 저장합니다.
고정 development에서 비교한 뒤 결과를 저장하고 종료합니다. **전체 학습은 자동으로 이어서 실행하지 않습니다.**
원본 Training 표본과 정답을 유지합니다. 9개 밖의 라벨만 제외하며 해당 통화는 남깁니다.
모델 특징은 배열 순서의 `utterances[].text`뿐이며 줄바꿈으로 발화 경계를 유지합니다.

**기본 180분 작업 예산에는 설치·다운로드·준비·속도 측정·학습·평가·저장이 모두 포함됩니다.**
Drive 로그인 대기 시간은 제외합니다. `time_budget_minutes`로 실행 전에 직접 늘릴 수 있고 자동 연장하지 않습니다.
**시간 제한 없이 실행하려면 `SETTINGS["time_budget_minutes"] = None`으로 설정하세요.**
이때 예상 시간에 따른 시작 보류, 설치·실행 시간 제한, 시간 만료 GPU 반환 타이머를 끕니다.
완료·오류 시 결과 기록과 GPU 반환은 유지하며, Colab 자체의 세션 종료는 막을 수 없습니다.
동일한 대표 표본으로 배치 후보들의 속도와 메모리를 측정한 뒤 현재 GPU에서 가장 빠른 성공 후보를 선택합니다.
시간 제한을 켠 경우 마지막 25분은 저장·재로딩·오류 보고·패키지 검증에 예약합니다.
예산이 부족하면 학습 시작을 보류하거나 미완료 상태로 저장합니다. 각 실행은 추가 GPU 리소스를 소비합니다.
**변경된 자동 배치 선택과 예비 학습의 GPU 실행 시간·F1 개선은 아직 확인되지 않았습니다.**


## 고정 실험 조건

| 항목 | Qwen 후보 | 새 KLUE 대조군 |
|---|---|---|
| 시작 가중치 | Qwen3-4B-Instruct-2507 | klue/roberta-base |
| 학습 | NF4 4-bit, double quantization, LoRA + 분류층 | 전체 encoder + 분류층 |
| 입력 길이 | 1,024, 앞/뒤 보존 + 공통 판정 접미문 | 특수 토큰 포함 앞부분 512 |
| 자동 측정 batch 후보 | 1 / 2 / 4 / 8 | 4 / 8 / 16 |
| 유효 batch | 16 (누적 횟수 자동 조정) | 16 (누적 횟수 자동 조정) |
| 학습률 | LoRA 1e-4 / head 3e-4 | 2e-5 |

9개 독립 logits에서 weighted BCE를 직접 계산합니다.
`pos_weight = clip(sqrt(음성 수 / max(양성 수, 1)), 1, 5)`는 실제 예비 학습 subset만으로 계산하고,
최종 refit에서는 전체 Training으로 다시 계산합니다. 판정은 항상 `logit >= 0`입니다.
LoRA r=16, alpha=32, dropout=0.05, attention/MLP projection 전체를 학습합니다.
양자화 원본은 고정합니다. bf16 미지원 GPU의 fp16 경로는 pilot에서 먼저 시험합니다.

정규화한 텍스트 중복 그룹을 보존하여 Training을 새 **70% fit / 15% development / 15% confirmation**으로 나눕니다.
라벨 비율은 가능한 한 유지하며 실제 분할과 데이터 digest를 저장합니다.
과거 전체 Training 모델은 이 비교에 사용하지 않습니다. 새로운 외부 검증셋이라는 뜻도 아닙니다.
기본 예비 실행은 1 epoch이며 같은 subset으로 학습한 두 모델을 고정 development에서 비교합니다.
소규모 1 epoch 결과만으로 전체 데이터 성능이나 수렴 여부를 단정하지 않습니다.
Qwen 또는 사전에 정한 1:1 확률 평균이 대조군보다 0.01 이상 높으면 추진 후보로 삼습니다.
0.01은 자원 배분 기준이며 통계적 유의성이나 목표 점수 보장이 아닙니다.
**Macro F1이 우선이며 오심 소폭 하락만으로 후보를 자동 탈락시키지 않습니다.**

**예비 실행에서는 confirmation을 평가하지 않고 공식 Validation을 모델 평가에 사용하지 않습니다.**
결과를 검토한 뒤 별도 full 실행을 시작하면 예비 실행에서 동결한 선택을 confirmation에서 한 번 비교합니다.
이 점수로 후보를 다시 고르지 않으며 선택 recipe를 공개 초기 가중치에서 전체 Training으로 새로 학습합니다.
9개 F1/AP, 오심 P/R/TP/FP/FN, 공통 오답·새 FP/FN, 통화 단위 bootstrap을 저장합니다.
공식 Validation은 후보 선택에 사용하지 않고 전체 Training refit 이후 한 번 확인합니다.
이미 반복 관찰된 공식 Validation 점수를 완전히 독립된 일반화 평가라고 부르지 않습니다.
threshold/offset/calibration/혼합비 검색, 재라벨링, 삭제, 정답 마스킹은 하지 않습니다.
첫 실행은 **weighted BCE만** 사용합니다. ASL 재학습과 3-fold OOF는 후속 실험이며 자동 실행하지 않습니다.
친구 11개 모델 앙상블은 보존할 별도 baseline이며 이번 새 KLUE 대조군과 다릅니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver25', 'run_mode': 'pilot', 'pilot_samples': 4000, 'source_run': '', 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver25', 'work_dir': '/content/mission3_ver25', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'fit_fraction': 0.7, 'development_fraction': 0.15, 'confirmation_fraction': 0.15, 'qwen_batch_size': 1, 'qwen_gradient_accumulation': 16, 'qwen_max_length': 1024, 'control_batch_size': 8, 'control_gradient_accumulation': 2, 'control_max_length': 512, 'max_epochs': 1, 'benchmark_examples': 64, 'benchmark_warmup_examples': 16, 'effective_batch_size': 16, 'warmup_ratio': 0.05, 'lora_lr': 0.0001, 'head_lr': 0.0003, 'control_lr': 2e-05, 'weight_decay': 0.01, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'gradient_clip': 1.0, 'min_macro_gain': 0.01, 'allow_cpu': False, 'time_budget_minutes': 180, 'finalization_reserve_seconds': 1500, 'budget_safety_factor': 1.35, 'log_every': 100}
# First run: keep run_mode='pilot'. Later: set run_mode='full', source_run to
# the completed pilot run folder. Set time_budget_minutes=None for no time limit,
# or use a positive integer from its time estimate for a bounded run.
AUTO_RELEASE_GPU = True # @param {type:"boolean"}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
# Each rerun starts a new budget and consumes additional Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 설정한 작업 예산 시작 (이후 패키지 설치 시간도 포함)
import time
import threading
import math
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if not isinstance(globals().get("SETTINGS"), dict) or "AUTO_RELEASE_GPU" not in globals():
    raise RuntimeError("실행 설정이 없습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
configured_minutes = SETTINGS["time_budget_minutes"]
if configured_minutes is not None and (
        isinstance(configured_minutes, bool) or not isinstance(configured_minutes, (int, float))
        or not math.isfinite(configured_minutes) or configured_minutes <= 0):
    raise ValueError("time_budget_minutes must be a finite positive number or None")
BUDGET_STARTED_MONOTONIC = time.monotonic()
DEADLINE_MONOTONIC = (None if configured_minutes is None else
                      BUDGET_STARTED_MONOTONIC + float(configured_minutes) * 60)
SETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC
SETTINGS["invocation_started_monotonic"] = BUDGET_STARTED_MONOTONIC
previous_timer = globals().get("RELEASE_TIMER")
if previous_timer is not None:
    previous_timer.cancel()
RELEASE_TIMER = None
RELEASE_LOCK = threading.Lock()
RELEASE_REQUESTED = False

def save_setup_failure(error, stage):
    # Never record exception text: it can contain input text or private paths.
    import json
    import traceback
    import uuid
    from datetime import datetime, timezone
    from pathlib import Path
    frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,
               "function": frame.f_code.co_name}
              for frame, line in traceback.walk_tb(error.__traceback__)]
    report = {"status": "failed", "stage": stage, "error_type": type(error).__name__,
              "category": "preparation_failed", "frames": frames[-20:]}
    print("준비 실패 단계:", stage, "· 오류 종류:", type(error).__name__, flush=True)
    try:
        folder = Path(SETTINGS["output_dir"]) / "_diagnostics"
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / ("ver25_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                         + "_" + uuid.uuid4().hex[:8] + ".json")
        path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        print("오류 보고서:", path, flush=True)
    except OSError as save_error:
        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, flush=True)

def release_gpu():
    global RELEASE_REQUESTED
    if not AUTO_RELEASE_GPU:
        return
    with RELEASE_LOCK:
        if RELEASE_REQUESTED:
            return
        RELEASE_REQUESTED = True
        if RELEASE_TIMER is not None:
            RELEASE_TIMER.cancel()
    try:
        from google.colab import runtime as colab_runtime
        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)
        colab_runtime.unassign()
    except Exception as error:
        print("자동 반환 요청 실패:", type(error).__name__,
              "· Colab 메뉴 '런타임 → 연결 해제 및 런타임 삭제'를 직접 실행하세요.", flush=True)

if AUTO_RELEASE_GPU and DEADLINE_MONOTONIC is not None:
    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)
    RELEASE_TIMER.daemon = True
    RELEASE_TIMER.start()
if DEADLINE_MONOTONIC is None:
    print("작업 시간 제한: 없음 · 완료 또는 오류 시 결과 기록 후 GPU 반환")
else:
    print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")


In [ ]:
# @title 실행 코드 준비 (코드 파일 추가 업로드 불필요)
import importlib.util
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Bounded Qwen QLoRA versus fresh KLUE multi-label experiment."""\n', 'data.py': '"""Text-only data and label-balanced, indivisible Training transcript groups."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        texts.append(utterance["text"])\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _assign_groups(counts, sizes, proportions, seed):\n    """Iteratively allocate the rarest remaining group label, weighted by rows.\n\n    Positive and negative counts are balanced together. A transcript group is\n    indivisible, including when its original rows have conflicting labels.\n    """\n    rng = np.random.default_rng(seed)\n    presence = counts > 0\n    remaining = presence.sum(axis=0)\n    label_deficits = proportions[:, None] * counts.sum(axis=0)\n    row_deficits = proportions * sizes.sum()\n    active = np.ones(len(sizes), dtype=bool)\n    assignment = np.full(len(sizes), -1, dtype=np.int8)\n    coverage = np.zeros((len(proportions), counts.shape[1]), dtype=bool)\n    assigned_counts = np.zeros(len(proportions), dtype=int)\n    unassigned_count = len(sizes)\n    while unassigned_count:\n        available = np.flatnonzero(remaining > 0)\n        rare = available[remaining[available] == remaining[available].min()]\n        label = int(rng.choice(rare))\n        candidates = np.flatnonzero(active & presence[:, label])\n        # Large indivisible groups are placed while there is still capacity.\n        rng.shuffle(candidates)\n        candidates = candidates[np.argsort(-sizes[candidates], kind="stable")]\n        for group in candidates:\n            sides = np.flatnonzero(row_deficits > 0)\n            if not len(sides):\n                sides = np.arange(len(proportions))\n            missing = np.flatnonzero(~coverage[:, label])\n            if len(missing) and remaining[label] <= len(missing):\n                sides = missing\n            # Every split must contain a group, even for very small datasets.\n            empty = np.flatnonzero(assigned_counts == 0)\n            if unassigned_count == len(empty):\n                sides = empty\n            sides = sides[label_deficits[sides, label] == label_deficits[sides, label].max()]\n            sides = sides[row_deficits[sides] == row_deficits[sides].max()]\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            row_deficits[side] -= sizes[group]\n            label_deficits[side] -= counts[group]\n            remaining -= presence[group]\n            coverage[side] |= presence[group]\n            active[group] = False\n            assigned_counts[side] += 1\n            unassigned_count -= 1\n    return assignment\n\n\ndef dataset_digest(texts, targets):\n    """Hash ordered exact model inputs and original nine-column targets."""\n    targets = _binary(targets, len(texts), "targets")\n    digest = hashlib.sha256()\n    digest.update(json.dumps(TARGET_SYMPTOMS, ensure_ascii=False).encode("utf-8"))\n    for text, row in zip(texts, targets):\n        encoded = text.encode("utf-8")\n        digest.update(len(encoded).to_bytes(8, "little"))\n        digest.update(encoded)\n        digest.update(row.tobytes())\n    return digest.hexdigest()\n\n\ndef three_way_split(texts, targets, seed: int = 42):\n    """Return fit/development/confirmation indices and an aggregate manifest.\n\n    Eight deterministic allocation attempts minimize only split label/row\n    imbalance. Model predictions and official Validation never participate.\n    No row or original label is dropped, masked, or changed.\n    """\n    targets = _binary(targets, len(texts), "targets")\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    if len(unique) < 3:\n        raise ValueError("Three-way split requires at least three distinct normalized text groups")\n    positive = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int64)\n    np.add.at(positive, inverse, targets)\n    counts = np.concatenate([positive, sizes[:, None] - positive], axis=1)\n    proportions = np.asarray([.70, .15, .15])\n    totals = counts.sum(axis=0)\n    best = None\n    for attempt in range(8):\n        assignment = _assign_groups(counts, sizes, proportions, seed + attempt)\n        splits = tuple(np.flatnonzero(assignment[inverse] == side) for side in range(3))\n        observed = np.asarray([counts[assignment == side].sum(axis=0) for side in range(3)])\n        missing = int(((observed == 0) & (totals > 0)).sum())\n        imbalance = float(np.abs(observed / np.maximum(totals, 1) - proportions[:, None]).mean()\n                          + np.abs(np.asarray([len(ids) for ids in splits]) / len(texts) - proportions).mean())\n        rank = (missing, imbalance)\n        if best is None or rank < best[0]:\n            best = (rank, splits, assignment, attempt)\n    _, splits, assignment, attempt = best\n    data_hash = dataset_digest(texts, targets)\n    split_hash = hashlib.sha256(data_hash.encode("ascii") + assignment[inverse].tobytes()).hexdigest()\n    report = {\n        "method": "iterative multilabel stratification of indivisible normalized text groups",\n        "group_normalization": "NFC and whitespace collapse; model input text is preserved",\n        "seed": seed, "allocation_attempt": attempt, "label_order": list(TARGET_SYMPTOMS),\n        "data_digest": data_hash, "split_digest": split_hash,\n        "samples": len(texts), "text_groups": len(unique),\n        "duplicate_groups": int((sizes > 1).sum()),\n        "duplicate_extra_samples": int((sizes - 1).sum()),\n        "conflicting_label_groups": int(np.any((positive > 0) & (positive < sizes[:, None]), axis=1).sum()),\n        "all_rows_preserved": True, "cross_split_text_groups": 0,\n        "positive_counts": dict(zip(TARGET_SYMPTOMS, targets.sum(axis=0).astype(int).tolist())),\n        "splits": {}, "coverage_warnings": [],\n        "limitations": "Group constraints may prevent exact fractions or label coverage. Near duplicates are not checked. "\n                       "Training has prior experiment exposure; this is not a new external test set.",\n    }\n    for side, (name, ids) in enumerate(zip(("fit", "development", "confirmation"), splits)):\n        positives = targets[ids].sum(axis=0)\n        report["splits"][name] = {\n            "samples": len(ids), "text_groups": int((assignment == side).sum()),\n            "requested_fraction": float(proportions[side]), "actual_fraction": len(ids) / len(texts),\n            "indices_digest": hashlib.sha256(ids.astype("<i8").tobytes()).hexdigest(),\n            "group_digest": hashlib.sha256("".join(unique[assignment == side]).encode("ascii")).hexdigest(),\n            "positive_counts": dict(zip(TARGET_SYMPTOMS, positives.astype(int).tolist())),\n            "negative_counts": dict(zip(TARGET_SYMPTOMS, (len(ids) - positives).astype(int).tolist())),\n            "positive_prevalence": dict(zip(TARGET_SYMPTOMS, (positives / len(ids)).tolist())),\n        }\n        for index, symptom in enumerate(TARGET_SYMPTOMS):\n            if positives[index] in (0, len(ids)):\n                report["coverage_warnings"].append({"split": name, "symptom": symptom,\n                    "missing_class": "positive" if positives[index] == 0 else "negative"})\n    return *splits, report\n\n\ndef _subset_groups_at_capacity(assignment, sizes, capacity, seed):\n    """Keep the preferred groups first, while reaching the largest feasible size."""\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(len(sizes))\n    order = order[np.argsort(assignment[order], kind="stable")]\n    # Bit i indicates whether i rows can be formed from whole remaining groups.\n    # Suffix reachability lets us preserve preferred groups without underfilling\n    # a feasible request merely because one large group was selected first.\n    mask = (1 << (capacity + 1)) - 1\n    reachable = [0] * (len(order) + 1)\n    reachable[-1] = 1\n    for position in range(len(order) - 1, -1, -1):\n        tail = reachable[position + 1]\n        reachable[position] = (tail | (tail << int(sizes[order[position]]))) & mask\n    remaining = reachable[0].bit_length() - 1\n    selected = np.zeros(len(sizes), dtype=bool)\n    for position, group in enumerate(order):\n        after = remaining - int(sizes[group])\n        if after >= 0 and (reachable[position + 1] >> after) & 1:\n            selected[group] = True\n            remaining = after\n    return selected\n\n\ndef sample_fit_subset(texts, targets, fit, sample_count: int = 4000, seed: int = 42):\n    """Select a reproducible, label-balanced pilot from fit groups only.\n\n    Original texts, targets, and the existing three-way split are unchanged.\n    If indivisible duplicates prevent the exact request, return the largest\n    feasible count below it and record the shortfall. No model predictions or\n    development/confirmation labels influence the allocation.\n    """\n    targets = _binary(targets, len(texts), "targets")\n    if type(sample_count) is not int or sample_count <= 0:\n        raise ValueError("sample_count must be a positive integer")\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    fit = np.asarray(fit)\n    if (fit.ndim != 1 or not np.issubdtype(fit.dtype, np.integer)\n            or len(np.unique(fit)) != len(fit)\n            or np.any(fit < 0) or np.any(fit >= len(texts))):\n        raise ValueError("fit must contain unique valid integer indices")\n    if len(fit) < sample_count:\n        raise ValueError("sample_count exceeds the available fit samples")\n    fit = np.sort(fit.astype(np.int64))\n    groups = np.asarray([text_group(text) for text in texts])\n    all_groups, all_inverse, all_sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    fit_sizes = np.bincount(all_inverse[fit], minlength=len(all_groups))\n    if np.any((fit_sizes > 0) & (fit_sizes != all_sizes)):\n        raise ValueError("fit must contain complete normalized text groups to prevent split leakage")\n    unique, inverse, sizes = np.unique(groups[fit], return_inverse=True, return_counts=True)\n    positive = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int64)\n    np.add.at(positive, inverse, targets[fit])\n    counts = np.concatenate([positive, sizes[:, None] - positive], axis=1)\n    parent_positive = targets[fit].sum(axis=0)\n    parent_prevalence = parent_positive / len(fit)\n    parent_counts = counts.sum(axis=0)\n    proportions = np.asarray([sample_count / len(fit), 1 - sample_count / len(fit)])\n    best = None\n    for attempt in range(8):\n        if sample_count == len(fit):\n            preferred = np.zeros(len(sizes), dtype=np.int8)\n        else:\n            preferred = _assign_groups(counts, sizes, proportions, seed + attempt)\n        selected = _subset_groups_at_capacity(preferred, sizes, sample_count, seed + attempt)\n        indices = fit[selected[inverse]]\n        if not len(indices):\n            raise ValueError("No complete fit text group fits within sample_count")\n        observed = counts[selected].sum(axis=0)\n        missing = int(((observed == 0) & (parent_counts > 0)).sum())\n        deviation = float(np.abs(targets[indices].mean(axis=0) - parent_prevalence).mean())\n        rank = (missing, deviation)\n        if best is None or rank < best[0]:\n            best = (rank, indices, selected, attempt)\n    _, indices, selected, attempt = best\n    data_hash = dataset_digest(texts, targets)\n    parent_hash = hashlib.sha256(fit.astype("<i8").tobytes()).hexdigest()\n    indices_hash = hashlib.sha256(indices.astype("<i8").tobytes()).hexdigest()\n    subset_hash = hashlib.sha256((data_hash + parent_hash + indices_hash).encode("ascii")).hexdigest()\n    observed_positive = targets[indices].sum(axis=0)\n    observed_prevalence = observed_positive / len(indices)\n    report = {\n        "method": "iterative multilabel stratification within fit, with exact whole-group capacity",\n        "group_normalization": "NFC and whitespace collapse; model input text is preserved",\n        "seed": seed, "allocation_attempt": attempt, "label_order": list(TARGET_SYMPTOMS),\n        "requested_samples": sample_count, "actual_samples": len(indices),\n        "parent_fit_samples": len(fit), "unfilled_samples": sample_count - len(indices),\n        "exact_sample_count": len(indices) == sample_count,\n        "data_digest": data_hash, "parent_fit_digest": parent_hash,\n        "indices_digest": indices_hash, "subset_digest": subset_hash,\n        "parent_fit_text_groups": len(unique), "selected_text_groups": int(selected.sum()),\n        "selected_duplicate_groups": int(((sizes > 1) & selected).sum()),\n        "selected_conflicting_label_groups": int((np.any(\n            (positive > 0) & (positive < sizes[:, None]), axis=1) & selected).sum()),\n        "outside_fit_samples": 0, "original_targets_preserved": True,\n        "positive_counts": dict(zip(TARGET_SYMPTOMS, observed_positive.astype(int).tolist())),\n        "negative_counts": dict(zip(TARGET_SYMPTOMS, (len(indices) - observed_positive).astype(int).tolist())),\n        "positive_prevalence": dict(zip(TARGET_SYMPTOMS, observed_prevalence.tolist())),\n        "parent_positive_counts": dict(zip(TARGET_SYMPTOMS, parent_positive.astype(int).tolist())),\n        "parent_positive_prevalence": dict(zip(TARGET_SYMPTOMS, parent_prevalence.tolist())),\n        "prevalence_deviation": dict(zip(TARGET_SYMPTOMS,\n                                         (observed_prevalence - parent_prevalence).tolist())),\n        "coverage": {}, "coverage_warnings": [],\n        "limitations": "Indivisible groups may prevent exact sample counts or class coverage. "\n                       "This pilot subset does not establish full-data performance.",\n    }\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        positive_present = bool(observed_positive[index] > 0)\n        negative_present = bool(observed_positive[index] < len(indices))\n        report["coverage"][symptom] = {"positive": positive_present, "negative": negative_present}\n        if not positive_present and parent_positive[index] > 0:\n            report["coverage_warnings"].append({"symptom": symptom, "missing_class": "positive"})\n        if not negative_present and parent_positive[index] < len(fit):\n            report["coverage_warnings"].append({"symptom": symptom, "missing_class": "negative"})\n    return indices, report\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\n', 'analysis.py': '"""Fixed-decision multilabel scores and paired call-level uncertainty."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nfrom .data import TARGET_SYMPTOMS, _binary, score_predictions\n\n\ndef _logits(values, rows=None):\n    values = np.asarray(values, dtype=np.float64)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and len(values) != rows) or not np.isfinite(values).all()):\n        raise ValueError("logits must be a finite (samples, 9) array aligned with targets")\n    return values\n\n\ndef metrics(targets, logits):\n    """Every logit >= 0 is positive, including exact zero ties."""\n    targets = _binary(targets, name="targets")\n    logits = _logits(logits, len(targets))\n    predicted = logits >= 0\n    result = score_predictions(targets, predicted)\n    ap_values = []\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        truth, prediction = targets[:, index].astype(bool), predicted[:, index]\n        row = result["per_symptom"][symptom]\n        # AP is undefined without positive targets; JSON null makes this explicit.\n        ap = float(average_precision_score(truth, logits[:, index])) if truth.any() else None\n        row.update({"true_positive": int((truth & prediction).sum()),\n                    "true_negative": int((~truth & ~prediction).sum()), "average_precision": ap})\n        if ap is not None:\n            ap_values.append(ap)\n    result["macro_average_precision"] = float(np.mean(ap_values)) if ap_values else None\n    result["average_precision_valid_labels"] = len(ap_values)\n    result["nausea_f1"] = result["per_symptom"]["오심"]["f1"]\n    result["decision_rule"] = "independent logits >= 0 (sigmoid >= 0.5); zero ties are positive"\n    return result\n\n\ndef ensemble_logits(control_logits, candidate_logits):\n    """Return logit of the fixed 1:1 sigmoid probability mean.\n\n    Compute the probability mean in log space, preserving its ranking for AP.\n    Its mathematical decision is a+b >= 0. Roundoff in the log-space\n    subtraction is repaired with that identity, without a fitted offset.\n    Same-sign inputs use their signs directly; opposite-sign addition cannot\n    overflow and retains the decision even for subnormal floating-point inputs.\n    """\n    left = _logits(control_logits)\n    right = _logits(candidate_logits, len(left))\n    log_positive = np.logaddexp(-np.logaddexp(0., -left), -np.logaddexp(0., -right))\n    log_negative = np.logaddexp(-np.logaddexp(0., left), -np.logaddexp(0., right))\n    mixed = log_positive - log_negative\n    same_sign = np.signbit(left) == np.signbit(right)\n    decision = np.sign(left) + np.sign(right)\n    decision[~same_sign] = left[~same_sign] + right[~same_sign]\n    nonzero = decision != 0\n    mixed[nonzero] = np.copysign(np.maximum(np.abs(mixed[nonzero]), np.nextafter(0., 1.)), decision[nonzero])\n    mixed[~nonzero] = 0.\n    return mixed\n\n\ndef paired_error_changes(targets, control_logits, candidate_logits):\n    """Compare paired rows; common errors identify remaining complementarity."""\n    targets = _binary(targets, name="targets").astype(bool)\n    control = _logits(control_logits, len(targets)) >= 0\n    candidate = _logits(candidate_logits, len(targets)) >= 0\n    if not len(targets):\n        raise ValueError("Cannot compare an empty dataset")\n    result = {"samples": len(targets), "per_symptom": {}}\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        truth, old, new = targets[:, index], control[:, index], candidate[:, index]\n        result["per_symptom"][symptom] = {\n            "common_false_negative": int((truth & ~old & ~new).sum()),\n            "common_false_positive": int((~truth & old & new).sum()),\n            "fixed_false_negative": int((truth & ~old & new).sum()),\n            "fixed_false_positive": int((~truth & old & ~new).sum()),\n            "new_false_negative": int((truth & old & ~new).sum()),\n            "new_false_positive": int((~truth & ~old & new).sum()),\n            "control_false_negative": int((truth & ~old).sum()),\n            "control_false_positive": int((~truth & old).sum()),\n            "candidate_false_negative": int((truth & ~new).sum()),\n            "candidate_false_positive": int((~truth & new).sum()),\n        }\n    return result\n\n\ndef _f1_from_counts(tp, fp, fn):\n    denominator = 2 * tp + fp + fn\n    return np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator != 0)\n\n\ndef paired_bootstrap_ci(targets, control_logits, candidate_logits, iterations: int = 500,\n                        seed: int = 42, confidence: float = .95):\n    """Percentile CIs for candidate-minus-control F1, resampling entire calls.\n\n    Both models and all nine labels use the identical resampled row indices.\n    This estimates uncertainty conditional on these trained models and split;\n    it does not include training seed or repeated selection uncertainty.\n    """\n    truth = _binary(targets, name="targets").astype(bool)\n    old = _logits(control_logits, len(truth)) >= 0\n    new = _logits(candidate_logits, len(truth)) >= 0\n    if not len(truth):\n        raise ValueError("Cannot bootstrap an empty dataset")\n    if type(iterations) is not int or iterations < 2:\n        raise ValueError("iterations must be an integer >= 2")\n    if type(seed) is not int or seed < 0 or not 0 < confidence < 1:\n        raise ValueError("seed must be nonnegative and confidence must be in (0, 1)")\n    # One row contains every class and both predictions: a call is the unit.\n    counts = np.stack([np.stack([truth & prediction, ~truth & prediction, truth & ~prediction], axis=1)\n                       for prediction in (old, new)], axis=1).astype(np.int8)\n    point = _f1_from_counts(*counts.sum(axis=0).transpose(1, 0, 2))\n    point_delta = point[1] - point[0]\n    rng = np.random.default_rng(seed)\n    delta = np.empty((iterations, len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for iteration in range(iterations):\n        sampled = counts[rng.integers(0, len(truth), size=len(truth))].sum(axis=0)\n        f1 = _f1_from_counts(*sampled.transpose(1, 0, 2))\n        delta[iteration] = f1[1] - f1[0]\n    quantiles = [(1 - confidence) / 2, (1 + confidence) / 2]\n\n    def interval(values, estimate):\n        lower, upper = np.quantile(values, quantiles)\n        return {"estimate": float(estimate), "lower": float(lower), "upper": float(upper)}\n\n    return {"samples": len(truth), "iterations": iterations, "seed": seed, "confidence": confidence,\n            "unit": "call", "comparison": "candidate minus control", "method": "paired percentile bootstrap",\n            "macro_f1_delta": interval(delta.mean(axis=1), point_delta.mean()),\n            "per_symptom_f1_delta": {name: interval(delta[:, index], point_delta[index])\n                                     for index, name in enumerate(TARGET_SYMPTOMS)},\n            "limitations": "Conditional on trained models and this split; repeated selection and training-seed "\n                           "uncertainty are not included. Duplicate calls may still be statistically dependent."}\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision enforces the actual invocation deadline. A ``None`` deadline\ndisables time limits while retaining normal completion and error handling.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    A ``None`` deadline admits work without a time limit and reports ``None``\n    remaining, so saved reports stay valid JSON without infinity values.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = (None if deadline_monotonic is None else\n                                   _finite_nonnegative(deadline_monotonic, "deadline_monotonic"))\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        if self.deadline_monotonic is None:\n            return None\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining is not None and remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining is None or (remaining > 0 and estimate * self.safety_factor <= remaining)\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and an optional wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    ``None`` waits for completion without a watchdog; error cleanup stays bounded.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = (None if deadline_monotonic is None else\n                _finite_nonnegative(deadline_monotonic, "deadline_monotonic"))\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline is not None and deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        timeout = None if deadline is None else max(0.0, deadline - margin - time.monotonic())\n        returncode = process.wait(timeout=timeout)\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline if deadline is not None else time.monotonic() + margin)\n        if deadline is None:\n            raise\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        cleanup_deadline = time.monotonic() + margin\n        _stop_process(process, cleanup_deadline if deadline is None else min(deadline, cleanup_deadline))\n        raise\n    if returncode:\n        cleanup_deadline = time.monotonic() + margin\n        _stop_process(process, cleanup_deadline if deadline is None else min(deadline, cleanup_deadline))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'batching.py': '"""Measure fresh-model batch candidates while keeping effective batch 16 fixed."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport math\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom . import data, trainer\n\nEFFECTIVE_BATCH = 16\nWARMUP_EXAMPLES = 16\nMEASURED_EXAMPLES = 64\nPREDICTION_EXAMPLES = 64\nMIN_HEADROOM_FRACTION = .10\nBATCH_CANDIDATES = {"qwen": (8, 4, 2, 1), "control": (16, 8, 4)}\n\n\nclass _TrainingBudget:\n    """Benchmark inference also spends only the time available before export."""\n    def __init__(self, budget):\n        self.budget = budget\n\n    def check(self, stage, training=False):\n        return self.budget.check(stage, training=True)\n\n\ndef _indices(values, rows, minimum, name):\n    values = np.asarray(values)\n    if (values.ndim != 1 or len(values) < minimum\n            or not np.issubdtype(values.dtype, np.integer)\n            or np.any(values < 0) or np.any(values >= rows)\n            or len(np.unique(values)) != len(values)):\n        raise ValueError(f"{name} needs at least {minimum} unique valid row indices")\n    return values.astype(np.int64, copy=False)\n\n\ndef representative_indices(texts, indices, count, seed):\n    """Sample evenly across transcript lengths, then fix the batch ordering."""\n    ordered = np.asarray(sorted(map(int, indices), key=lambda row: (len(texts[row]), row)))\n    positions = np.floor((np.arange(count) + .5) * len(ordered) / count).astype(np.int64)\n    selected = ordered[positions]\n    return np.random.default_rng(seed).permutation(selected)\n\n\ndef _digest(indices):\n    return hashlib.sha256(np.asarray(indices, dtype="<i8").tobytes()).hexdigest()\n\n\ndef _release(device):\n    gc.collect()\n    if torch.device(device).type == "cuda":\n        torch.cuda.empty_cache()\n\n\ndef _memory(device):\n    if torch.device(device).type != "cuda":\n        return {"peak_vram_bytes": 0, "peak_reserved_bytes": 0,\n                "device_total_bytes": None, "headroom_bytes": None,\n                "headroom_fraction": None, "memory_safe": True}\n    trainer._synchronize(device)\n    free, total = torch.cuda.mem_get_info(device)\n    reserved = torch.cuda.memory_reserved(device)\n    peak_reserved = torch.cuda.max_memory_reserved(device)\n    # CUDA free excludes our reserved pool and other processes. Estimate free\n    # space at our peak as well, so cached pages are not counted twice.\n    headroom = min(free, free + reserved - peak_reserved)\n    return {"peak_vram_bytes": int(torch.cuda.max_memory_allocated(device)),\n            "peak_reserved_bytes": int(peak_reserved), "device_total_bytes": int(total),\n            "headroom_bytes": int(headroom), "headroom_fraction": headroom / total,\n            "memory_safe": headroom >= total * MIN_HEADROOM_FRACTION}\n\n\ndef _stress_loader(kind, tokenizer, text, target, batch_size, max_length):\n    """A full update at maximum length, including optimizer-state allocation.\n\n    Extra synthetic padding tokens are deliberately attended to: kernels that\n    unpad masked inputs must also exercise the configured maximum sequence.\n    These disposable weights are never kept for either experiment stage.\n    """\n    source = trainer._Collate(tokenizer, kind, max_length)([(text, target)])\n    tokens = source["input_ids"]\n    missing = max_length - tokens.shape[1]\n    tokens = torch.nn.functional.pad(tokens, (0, missing), value=tokenizer.pad_token_id)\n    batch = {"input_ids": tokens.repeat(batch_size, 1),\n             "attention_mask": torch.ones((batch_size, max_length), dtype=torch.long),\n             "labels": source["labels"].repeat(batch_size, 1)}\n    if "token_type_ids" in source:\n        batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])\n    return [batch] * (EFFECTIVE_BATCH // batch_size)\n\n\ndef _trial(kind, spec, texts, targets, fit, warmup, measured, development,\n           settings, device, budget, batch_size, root):\n    """Do not return models, tensors, optimizers, logits, or exception objects."""\n    budget.check("batch benchmark model load", training=True)\n    started = time.monotonic()\n    model, tokenizer = trainer.create_model(spec, settings, device)\n    setup_seconds = time.monotonic() - started\n    budget.check("batch benchmark model loaded", training=True)\n    accumulation = EFFECTIVE_BATCH // batch_size\n    maximum = settings[kind + "_max_length"]\n    lengths_path = Path(root) / (kind + "_token_lengths.json")\n    if not lengths_path.exists():\n        budget.check("Training token lengths", training=True)\n        data.write_json(lengths_path, trainer.length_statistics(tokenizer, texts, kind, maximum))\n        budget.check("Training token lengths complete", training=True)\n    optimizer = trainer.build_optimizer(model, kind, settings)\n    scheduler = trainer.build_scheduler(optimizer, 6, settings["warmup_ratio"])\n    scaler = trainer.make_scaler(device)\n    weights = trainer.positive_weights(targets[fit])\n    memory = []\n    common = dict(device=device, pos_weight=weights, accumulation_steps=accumulation,\n                  scheduler=scheduler, scaler=scaler, budget=budget,\n                  gradient_clip=settings.get("gradient_clip", 1.0))\n\n    def train(loader):\n        budget.check("batch benchmark training", training=True)\n        stats = trainer.train_epoch(model, loader, optimizer, **common)\n        memory.append(_memory(device))\n        if not stats["completed_epoch"]:\n            raise RuntimeError("Batch benchmark did not finish its fixed sample set")\n        return stats\n\n    def loader(indices, with_targets=True):\n        return trainer.make_loader(tokenizer, texts, targets if with_targets else None,\n            indices, kind=kind, batch_size=batch_size, max_length=maximum,\n            shuffle=False, seed=settings["seed"], pin_memory=torch.device(device).type == "cuda")\n\n    stress_row = max(map(int, fit), key=lambda row: len(texts[row]))\n    stress = train(_stress_loader(kind, tokenizer, texts[stress_row], targets[stress_row], batch_size, maximum))\n    warmup_stats = train(loader(warmup))\n    stats = train(loader(measured))\n    if (stress["examples"] != EFFECTIVE_BATCH or warmup_stats["examples"] != WARMUP_EXAMPLES\n            or stats["examples"] != MEASURED_EXAMPLES\n            or stats["successful_updates"] != MEASURED_EXAMPLES // EFFECTIVE_BATCH):\n        raise RuntimeError("Batch benchmark compared unequal training workloads")\n    prediction = trainer.predict_logits(model, loader(development, with_targets=False),\n        device=device, budget=_TrainingBudget(budget), return_stats=True)\n    memory.append(_memory(device))\n    if prediction["examples"] != PREDICTION_EXAMPLES:\n        raise RuntimeError("Batch benchmark compared unequal prediction workloads")\n    seconds = stats["elapsed_seconds"] / MEASURED_EXAMPLES\n    predict_seconds = prediction["seconds_per_example"]\n    if not all(math.isfinite(value) and value > 0 for value in (seconds, predict_seconds)):\n        raise ValueError("Batch benchmark timings must be finite and positive")\n    cuda_memory = [item for item in memory if item["device_total_bytes"] is not None]\n    safe = all(item["memory_safe"] for item in memory)\n    result = {"status": "success" if safe else "insufficient_memory_headroom",\n        "batch_size": batch_size, "gradient_accumulation": accumulation,\n        "effective_batch_size": EFFECTIVE_BATCH, "seconds_per_example": seconds,\n        "examples_per_second": 1 / seconds, "seconds_per_microstep": seconds * batch_size,\n        "seconds_per_prediction": predict_seconds, "setup_seconds": setup_seconds,\n        "measured_examples": MEASURED_EXAMPLES, "warmup_examples": WARMUP_EXAMPLES,\n        "stress_examples": EFFECTIVE_BATCH, "stress_max_length": maximum,\n        "measured_optimizer_updates": stats["successful_updates"],\n        "overflow_retries": sum(item.get("overflow_retries", 0) for item in (stress, warmup_stats, stats)),\n        "final_scaler_scale": stats.get("final_scaler_scale"),\n        "prediction_examples": PREDICTION_EXAMPLES,\n        "peak_vram_bytes": max(item["peak_vram_bytes"] for item in memory),\n        "peak_reserved_bytes": max(item["peak_reserved_bytes"] for item in memory),\n        "device_total_bytes": cuda_memory[0]["device_total_bytes"] if cuda_memory else None,\n        "headroom_fraction": min(item["headroom_fraction"] for item in cuda_memory) if cuda_memory else None,\n        "memory_check": "cuda_peak_reserved_and_free" if cuda_memory else "cpu_test_only",\n        "pilot_weights_discarded": True}\n    return result\n\n\ndef _is_oom(error):\n    return isinstance(error, torch.cuda.OutOfMemoryError) or (\n        isinstance(error, RuntimeError) and any(phrase in str(error).lower()\n            for phrase in ("cuda out of memory", "cuda error: out of memory")))\n\n\ndef autotune(kind, spec, texts, targets, fit, development, settings, device, budget, root):\n    """Return updated settings and a measured, auditable batch-selection report.\n\n    Candidate models always start from the pinned pretrained weights. Only the\n    supplied fit labels enter loss or positive weights; development is timed\n    without labels. A later caller creates another fresh model for training.\n    """\n    if kind not in BATCH_CANDIDATES or trainer._kind(spec) != kind:\n        raise ValueError("Batch benchmark requires matching control or qwen model kind")\n    targets = trainer._targets(targets, len(texts))\n    fit = _indices(fit, len(texts), WARMUP_EXAMPLES + MEASURED_EXAMPLES, "Training benchmark")\n    development = _indices(development, len(texts), PREDICTION_EXAMPLES, "Development benchmark")\n    if np.intersect1d(fit, development).size:\n        raise ValueError("Training and development benchmark rows must be disjoint")\n    ordered = representative_indices(texts, fit, WARMUP_EXAMPLES + MEASURED_EXAMPLES, settings["seed"])\n    warmup, measured = ordered[:WARMUP_EXAMPLES], ordered[WARMUP_EXAMPLES:]\n    prediction = representative_indices(texts, development, PREDICTION_EXAMPLES, settings["seed"])\n    report = {"kind": kind, "status": "running", "selection": "fastest_safe_seconds_per_example",\n        "effective_batch_size": EFFECTIVE_BATCH, "minimum_headroom_fraction": MIN_HEADROOM_FRACTION,\n        "candidate_batches": list(BATCH_CANDIDATES[kind]), "warmup_examples": WARMUP_EXAMPLES,\n        "measured_examples": MEASURED_EXAMPLES, "prediction_examples": PREDICTION_EXAMPLES,\n        "warmup_indices_digest": _digest(warmup), "measured_indices_digest": _digest(measured),\n        "prediction_indices_digest": _digest(prediction), "pilot_weights_discarded": True,\n        "trials": []}\n    path = Path(root) / (kind + "_batch_benchmark.json")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    data.write_json(path, report)\n    _release(device)\n    for batch_size in BATCH_CANDIDATES[kind]:\n        print(f"{kind}: measuring batch {batch_size} x accumulation {EFFECTIVE_BATCH // batch_size}", flush=True)\n        try:\n            trial = _trial(kind, spec, texts, targets, fit, warmup, measured, prediction,\n                           settings, device, budget, batch_size, root)\n        except BaseException as error:\n            if not _is_oom(error):\n                report["status"] = "interrupted"\n                report["error_type"] = type(error).__name__\n                data.write_json(path, report)\n                raise\n            trial = {"status": "out_of_memory", "batch_size": batch_size,\n                "gradient_accumulation": EFFECTIVE_BATCH // batch_size,\n                "effective_batch_size": EFFECTIVE_BATCH, "error_type": type(error).__name__,\n                "pilot_weights_discarded": True}\n            # Drop training frames (and their GPU tensors) BEFORE collecting or\n            # trying the next candidate. Never retain the raw OOM payload.\n            error.__traceback__ = None\n            error.__context__ = None\n            error.__cause__ = None\n        finally:\n            _release(device)\n        trial.update(warmup_indices_digest=report["warmup_indices_digest"],\n                     measured_indices_digest=report["measured_indices_digest"],\n                     prediction_indices_digest=report["prediction_indices_digest"])\n        report["trials"].append(trial)\n        data.write_json(path, report)\n    eligible = [trial for trial in report["trials"] if trial["status"] == "success"]\n    if not eligible:\n        report["status"] = "no_safe_candidate"\n        data.write_json(path, report)\n        raise RuntimeError(f"No safe {kind} batch candidate; inspect {path.name}")\n    selected = min(eligible, key=lambda trial: (trial["seconds_per_example"], trial["batch_size"]))\n    report.update({key: value for key, value in selected.items() if key != "status"})\n    report["status"] = "selected"\n    report["probe_examples"] = MEASURED_EXAMPLES\n    selected_settings = dict(settings)\n    selected_settings[kind + "_batch_size"] = selected["batch_size"]\n    selected_settings[kind + "_gradient_accumulation"] = selected["gradient_accumulation"]\n    data.write_json(path, report)\n    print(f"{kind}: selected batch {selected[\'batch_size\']}; {selected[\'examples_per_second\']:.2f} examples/s", flush=True)\n    return selected_settings, report\n', 'trainer.py': '"""Ver25 weighted-BCE KLUE control and last-token Qwen QLoRA classifier.\n\nOnly caller-selected Training rows enter the explicit loss. Quantized frozen\nbases are saved once before adapter injection; reload never needs the network.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import nullcontext\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup\n\nfrom .data import TARGET_SYMPTOMS\n\nFORMAT = "mission3_ver25_qlora_bce_v1"\nDECISION = "independent_logits_greater_than_or_equal_to_zero"\nJUDGMENT_SUFFIX = "\\n위 통화에서 환자에게 나타난 증상을 판단하세요."\nLORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")\n\n\ndef _get(settings, name, default=None):\n    return settings.get(name, default) if isinstance(settings, dict) else getattr(settings, name, default)\n\n\ndef _kind(spec):\n    value = _get(spec, "kind", _get(spec, "name"))\n    if value not in ("control", "qwen"):\n        raise ValueError("Model kind must be control or qwen")\n    return value\n\n\ndef seed_everything(seed=42):\n    random.seed(int(seed))\n    np.random.seed(int(seed))\n    torch.manual_seed(int(seed))\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(int(seed))\n\n\ndef _targets(values, rows=None):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and len(values) != rows)\n            or not np.isin(values, (0, 1)).all()):\n        raise ValueError("Targets must be a binary (samples, 9) array")\n    return values.astype(np.float32, copy=False)\n\n\ndef positive_weights(fit_targets):\n    values = _targets(fit_targets)\n    if not len(values):\n        raise ValueError("Positive weights require at least one fit row")\n    positives = values.sum(axis=0, dtype=np.float64)\n    ratios = (len(values) - positives) / np.maximum(positives, 1)\n    return torch.tensor(np.clip(np.sqrt(ratios), 1, 5), dtype=torch.float32)\n\n\ndef fixed_decisions(logits):\n    values = np.asarray(logits)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or not np.isfinite(values).all()):\n        raise ValueError("Logits must be finite with shape (samples, 9)")\n    return (values >= 0).astype(np.int8)\n\n\nclass MeanPoolClassifier(nn.Module):\n    def __init__(self, encoder, dropout=.1):\n        super().__init__()\n        if encoder.config.model_type not in ("bert", "roberta"):\n            raise ValueError("The control requires BERT or RoBERTa")\n        if not 0 <= float(dropout) < 1:\n            raise ValueError("Dropout must be in [0, 1)")\n        self.encoder = encoder\n        if hasattr(encoder, "pooler"):\n            encoder.pooler = None\n        self.dropout_probability = float(dropout)\n        self.dropout = nn.Dropout(self.dropout_probability)\n        self.classifier = nn.Linear(encoder.config.hidden_size, len(TARGET_SYMPTOMS))\n        self.kind = "control"\n\n    def forward(self, input_ids, attention_mask, token_type_ids=None):\n        inputs = dict(input_ids=input_ids, attention_mask=attention_mask)\n        if token_type_ids is not None:\n            inputs["token_type_ids"] = token_type_ids\n        hidden = self.encoder(**inputs).last_hidden_state\n        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)\n        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n        return self.classifier(self.dropout(pooled))\n\n\nclass LastTokenClassifier(nn.Module):\n    def __init__(self, encoder):\n        super().__init__()\n        if encoder.config.model_type != "qwen3":\n            raise ValueError("The Qwen candidate requires a Qwen3 backbone")\n        self.encoder = encoder\n        self.encoder.config.use_cache = False\n        # This independent head remains float32, trainable, and explicitly saved.\n        self.classifier = nn.Linear(encoder.config.hidden_size, len(TARGET_SYMPTOMS))\n        self.kind = "qwen"\n\n    def forward(self, input_ids, attention_mask, token_type_ids=None):\n        if attention_mask.ndim != 2 or (attention_mask.sum(1) == 0).any():\n            raise ValueError("Each example needs at least one unpadded token")\n        positions = attention_mask.long().cumsum(-1) - 1\n        positions.masked_fill_(attention_mask == 0, 0)\n        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask,\n                              position_ids=positions, use_cache=False,\n                              return_dict=True).last_hidden_state\n        indexes = torch.arange(attention_mask.shape[1], device=hidden.device)\n        last = indexes.unsqueeze(0).masked_fill(attention_mask == 0, -1).max(1).values\n        pooled = hidden[torch.arange(len(hidden), device=hidden.device), last]\n        return self.classifier(pooled.to(self.classifier.weight.dtype))\n\n\ndef create_model(spec, settings, device, local_files_only=False):\n    """Start each comparison/refit from the pinned public pretrained backbone."""\n    kind = _kind(spec)\n    model_id, revision = _get(spec, "model_id"), _get(spec, "revision")\n    source = _get(spec, "snapshot_path") or model_id\n    local = isinstance(source, (str, Path)) and Path(source).is_dir()\n    if model_id is None or source is None:\n        raise ValueError("A model_id and pretrained source are required")\n    if (not Path(str(model_id)).is_dir()\n            and (not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision))):\n        raise ValueError("Remote pretrained models require an immutable 40-character revision")\n    device = torch.device(device)\n    test_only = bool(_get(settings, "test_only_unquantized", False))\n    if test_only and not (local and local_files_only):\n        raise ValueError("Unquantized Qwen is restricted to explicit local offline tests")\n    if kind == "qwen" and not test_only and device.type != "cuda":\n        raise RuntimeError("The production NF4 QLoRA recipe requires CUDA")\n    seed_everything(_get(settings, "seed", 42))\n    options = dict(trust_remote_code=False, local_files_only=local_files_only)\n    if not local:\n        options["revision"] = revision\n    if _get(settings, "cache_dir"):\n        options["cache_dir"] = str(_get(settings, "cache_dir"))\n    tokenizer = AutoTokenizer.from_pretrained(str(source), use_fast=True, **options)\n    if tokenizer.pad_token_id is None:\n        if kind != "qwen" or tokenizer.eos_token_id is None:\n            raise ValueError("Tokenizer needs a pad token or a Qwen EOS padding fallback")\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    if kind == "control":\n        encoder = AutoModel.from_pretrained(str(source), add_pooling_layer=False, **options)\n        model = MeanPoolClassifier(encoder, _get(settings, "control_dropout", .1)).to(device)\n    else:\n        from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n        compute_dtype = _amp_dtype(device) or torch.float32\n        loading = dict(options)\n        if not test_only:\n            loading.update(quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=compute_dtype), device_map={"": str(device)},\n                torch_dtype=compute_dtype)\n        encoder = AutoModel.from_pretrained(str(source), **loading)\n        if encoder.config.model_type != "qwen3":\n            raise ValueError("The Qwen candidate requires Qwen3 weights")\n        encoder.config.pad_token_id = tokenizer.pad_token_id\n        encoder.config.use_cache = False\n        # Saving here is essential: after injection the raw module state includes adapters.\n        base_store = _get(settings, "base_store_dir")\n        if base_store is None:\n            raise ValueError("Qwen creation requires a shared base_store_dir for portable saves")\n        identity = {"model_id": str(model_id), "revision": revision,\n                    "quantized": not test_only, "compute_dtype": str(compute_dtype)}\n        base_directory = _save_frozen_base(Path(base_store), encoder, identity)\n        if test_only:\n            encoder.requires_grad_(False)\n            encoder.to(device)\n        else:\n            encoder = prepare_model_for_kbit_training(\n                encoder, use_gradient_checkpointing=True,\n                gradient_checkpointing_kwargs={"use_reentrant": False})\n        adapter = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,\n                             r=int(_get(settings, "lora_r", 16)),\n                             lora_alpha=int(_get(settings, "lora_alpha", 32)),\n                             lora_dropout=float(_get(settings, "lora_dropout", .05)),\n                             target_modules=list(LORA_TARGETS), bias="none")\n        encoder = get_peft_model(encoder, adapter)\n        model = LastTokenClassifier(encoder)\n        model.classifier.to(device)\n        model._base_directory = base_directory\n        model._base_identity = identity\n    if len(tokenizer) > model.encoder.config.vocab_size:\n        raise ValueError("Tokenizer vocabulary exceeds the model embedding vocabulary")\n    return model, tokenizer\n\n\nclass _Rows(Dataset):\n    def __init__(self, texts, targets=None, indices=None):\n        if not len(texts) or any(not isinstance(text, str) for text in texts):\n            raise ValueError("Texts must be a nonempty sequence of strings")\n        self.texts = texts\n        self.targets = None if targets is None else _targets(targets, len(texts))\n        self.indices = np.arange(len(texts), dtype=np.int64) if indices is None else np.asarray(indices)\n        if (self.indices.ndim != 1 or not len(self.indices)\n                or not np.issubdtype(self.indices.dtype, np.integer)\n                or np.any(self.indices < 0) or np.any(self.indices >= len(texts))\n                or len(np.unique(self.indices)) != len(self.indices)):\n            raise ValueError("Row indices must be unique valid integers")\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, index):\n        row = int(self.indices[index])\n        return self.texts[row], None if self.targets is None else self.targets[row]\n\n\ndef qwen_token_ids(tokenizer, text, max_length):\n    suffix = tokenizer.encode(JUDGMENT_SUFFIX, add_special_tokens=False)\n    available = max_length - len(suffix)\n    if available < 2:\n        raise ValueError("Max length must reserve the full fixed judgment suffix plus transcript tokens")\n    transcript = tokenizer.encode(text, add_special_tokens=False)\n    if len(transcript) > available:\n        head = (available + 1) // 2\n        tail = available - head\n        transcript = transcript[:head] + transcript[-tail:]\n    return transcript + suffix\n\n\nclass _Collate:\n    def __init__(self, tokenizer, kind, max_length):\n        self.tokenizer, self.kind, self.max_length = tokenizer, kind, max_length\n\n    def __call__(self, rows):\n        if self.kind == "qwen":\n            inputs = [{"input_ids": qwen_token_ids(self.tokenizer, row[0], self.max_length)} for row in rows]\n            tokens = self.tokenizer.pad(inputs, padding=True, return_tensors="pt")\n        else:\n            tokens = self.tokenizer([row[0] for row in rows], padding=True,\n                                    truncation=True, max_length=self.max_length, return_tensors="pt")\n        result = {name: value for name, value in tokens.items()\n                  if name in ("input_ids", "attention_mask", "token_type_ids")}\n        if rows[0][1] is not None:\n            result["labels"] = torch.tensor(np.stack([row[1] for row in rows]), dtype=torch.float32)\n        return result\n\n\ndef make_loader(tokenizer, texts, targets=None, indices=None, *, kind, batch_size=8,\n                max_length=512, shuffle=False, seed=42, pin_memory=False):\n    if kind not in ("control", "qwen"):\n        raise ValueError("Model kind must be control or qwen")\n    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:\n        raise ValueError("Batch size must be a positive integer")\n    maximum = 1024 if kind == "qwen" else 512\n    if isinstance(max_length, bool) or not isinstance(max_length, int) or not 2 <= max_length <= maximum:\n        raise ValueError(f"Max length must be an integer in [2, {maximum}]")\n    if kind == "qwen":\n        qwen_token_ids(tokenizer, "", max_length)\n    generator = torch.Generator().manual_seed(int(seed))\n    return DataLoader(_Rows(texts, targets, indices), batch_size=batch_size,\n                      shuffle=bool(shuffle), generator=generator,\n                      collate_fn=_Collate(tokenizer, kind, max_length), drop_last=False,\n                      num_workers=0, pin_memory=bool(pin_memory))\n\n\ndef length_statistics(tokenizer, texts, kind, max_length):\n    if not texts or kind not in ("control", "qwen"):\n        raise ValueError("Length statistics require nonempty Training texts and a valid model kind")\n    suffix_length = len(tokenizer.encode(JUDGMENT_SUFFIX, add_special_tokens=False)) if kind == "qwen" else 0\n    lengths = np.asarray([len(tokenizer.encode(text, add_special_tokens=kind == "control")) + suffix_length\n                          for text in texts], dtype=np.int64)\n    truncated = np.maximum(lengths - max_length, 0)\n    return {"rows": len(texts), "max_length": max_length, "suffix_tokens": suffix_length,\n            "raw_token_percentiles": {str(q): float(np.percentile(lengths, q)) for q in (0, 25, 50, 75, 90, 95, 99, 100)},\n            "overlength_count": int((lengths > max_length).sum()),\n            "overlength_rate": float(np.mean(lengths > max_length)),\n            "removed_tokens": int(truncated.sum()),\n            "mean_effective_tokens": float(np.minimum(lengths, max_length).mean()),\n            "truncation": "transcript_head_tail_with_fixed_suffix" if kind == "qwen" else "right_truncation"}\n\n\ndef build_optimizer(model, kind, settings):\n    if kind not in ("control", "qwen"):\n        raise ValueError("Model kind must be control or qwen")\n    decay = float(_get(settings, "weight_decay", .01))\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        lr = float(_get(settings, "head_lr", 3e-4) if kind == "qwen" and name.startswith("classifier.")\n                   else _get(settings, "lora_lr", 1e-4) if kind == "qwen"\n                   else _get(settings, "control_lr", 2e-5))\n        wd = 0.0 if name.endswith("bias") or "norm" in name.lower() else decay\n        if not math.isfinite(lr) or lr <= 0 or not math.isfinite(wd) or wd < 0:\n            raise ValueError("Invalid optimizer settings")\n        groups.setdefault((lr, wd), []).append(parameter)\n    if not groups:\n        raise ValueError("The model has no trainable parameters")\n    return torch.optim.AdamW([{"params": parameters, "lr": lr, "weight_decay": wd}\n                              for (lr, wd), parameters in groups.items()])\n\n\ndef build_scheduler(optimizer, total_updates, warmup_ratio=.05):\n    if (isinstance(total_updates, bool) or not isinstance(total_updates, int)\n            or total_updates < 1 or not 0 <= warmup_ratio < 1):\n        raise ValueError("Invalid learning-rate schedule")\n    return get_linear_schedule_with_warmup(\n        optimizer, int(total_updates * warmup_ratio), total_updates)\n\n\ndef _amp_dtype(device):\n    if torch.device(device).type != "cuda":\n        return None\n    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n\n\ndef make_scaler(device):\n    device = torch.device(device)\n    return torch.amp.GradScaler("cuda", enabled=_amp_dtype(device) == torch.float16)\n\n\ndef _autocast(device):\n    dtype = _amp_dtype(device)\n    return torch.autocast("cuda", dtype=dtype) if dtype is not None else nullcontext()\n\n\ndef _synchronize(device):\n    if torch.device(device).type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef train_epoch(model, loader, optimizer, *, device, pos_weight, accumulation_steps=2,\n                scheduler=None, scaler=None, budget=None, on_update=None, max_updates=None,\n                max_microsteps=None, gradient_clip=1.0, max_overflow_retries=8):\n    """Train all rows once, including the final partial accumulation group.\n\n    A pilot can stop after ``max_updates`` successful updates. Such a partial\n    epoch is explicitly marked incomplete and must never be a final refit.\n    A recoverable fp16 scaler overflow replays the full accumulation group;\n    no Training rows or scheduler steps are lost. Timing includes replay work.\n    """\n    if (isinstance(accumulation_steps, bool) or not isinstance(accumulation_steps, int)\n            or accumulation_steps < 1):\n        raise ValueError("Accumulation steps must be a positive integer")\n    if max_updates is not None and (isinstance(max_updates, bool)\n                                    or not isinstance(max_updates, int) or max_updates < 1):\n        raise ValueError("Max updates must be a positive integer")\n    if max_microsteps is not None and (isinstance(max_microsteps, bool)\n            or not isinstance(max_microsteps, int) or max_microsteps < 1):\n        raise ValueError("Max microsteps must be a positive integer")\n    if not math.isfinite(gradient_clip) or gradient_clip <= 0:\n        raise ValueError("Gradient clipping must be finite and positive")\n    if (isinstance(max_overflow_retries, bool) or not isinstance(max_overflow_retries, int)\n            or max_overflow_retries < 0):\n        raise ValueError("Maximum overflow retries must be a nonnegative integer")\n    planned_batches = min(len(loader), max_microsteps or len(loader))\n    if not len(loader):\n        raise ValueError("Training loader is empty")\n    device = torch.device(device)\n    weights = torch.as_tensor(pos_weight, dtype=torch.float32, device=device)\n    if weights.shape != (len(TARGET_SYMPTOMS),) or not torch.isfinite(weights).all() or (weights <= 0).any():\n        raise ValueError("Positive weights must contain nine finite positive numbers")\n    scaler = make_scaler(device) if scaler is None else scaler\n    loss_function = nn.BCEWithLogitsLoss(pos_weight=weights)\n    model.train()\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n    optimizer.zero_grad(set_to_none=True)\n    successful = attempted = examples = seen_batches = group_rows = 0\n    loss_sum = 0.0\n    update_seconds = []\n    microstep_seconds = []\n    group_batches = []\n    group_rng = group_cuda_rng = None\n\n    def backward_batch(batch):\n        labels = batch["labels"].to(device, non_blocking=True)\n        inputs = {key: value.to(device, non_blocking=True) for key, value in batch.items()\n                  if key != "labels"}\n        with _autocast(device):\n            loss = loss_function(model(**inputs).float(), labels)\n        if not torch.isfinite(loss):\n            raise FloatingPointError("Training loss is not finite")\n        # Divide by total rows only after accumulation, including short tails.\n        scaler.scale(loss * len(labels)).backward()\n        return len(labels), float(loss.detach()) * len(labels)\n\n    _synchronize(device)\n    started = previous_update = previous_microstep = time.monotonic()\n    try:\n        for batch_index, batch in enumerate(loader):\n            if budget is not None:\n                budget.check("classifier_training", training=True)\n            if "labels" not in batch:\n                raise ValueError("Training batches must contain labels")\n            if scaler.is_enabled():\n                if not group_batches:\n                    group_rng = torch.get_rng_state()\n                    group_cuda_rng = torch.cuda.get_rng_state(device) if device.type == "cuda" else None\n                group_batches.append({key: value.detach().to("cpu", copy=True) for key, value in batch.items()})\n            rows, batch_loss_sum = backward_batch(batch)\n            group_rows += rows\n            examples += rows\n            seen_batches += 1\n            loss_sum += batch_loss_sum\n            if (batch_index + 1) % accumulation_steps and batch_index + 1 < planned_batches:\n                _synchronize(device)\n                microstep_finished = time.monotonic()\n                microstep_seconds.append(microstep_finished - previous_microstep)\n                previous_microstep = microstep_finished\n                continue\n            retries = 0\n            while True:\n                scaler.unscale_(optimizer)\n                for parameter in model.parameters():\n                    if parameter.grad is not None:\n                        parameter.grad.div_(group_rows)\n                norm = nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)\n                if not scaler.is_enabled() and not torch.isfinite(norm):\n                    raise FloatingPointError("Training gradients are not finite")\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                attempted += 1\n                accepted = scaler.get_scale() >= old_scale\n                optimizer.zero_grad(set_to_none=True)\n                if accepted:\n                    break\n                if retries >= max_overflow_retries:\n                    raise FloatingPointError("Gradient scaler overflow retry limit exceeded; no rows were skipped")\n                retries += 1\n                torch.set_rng_state(group_rng)\n                if group_cuda_rng is not None:\n                    torch.cuda.set_rng_state(group_cuda_rng, device)\n                for replay_batch in group_batches:\n                    if budget is not None:\n                        budget.check("classifier_overflow_replay", training=True)\n                    backward_batch(replay_batch)\n            group_batches.clear()\n            group_rows = 0\n            _synchronize(device)\n            now = time.monotonic()\n            microstep_seconds.append(now - previous_microstep)\n            previous_microstep = now\n            successful += 1\n            update_seconds.append(now - previous_update)\n            if scheduler is not None:\n                scheduler.step()\n            progress = {"successful_updates": successful, "attempted_updates": attempted,\n                        "overflow_retries": attempted - successful,\n                        "examples": examples, "batches": seen_batches, "microsteps": seen_batches,\n                        "elapsed_seconds": now - started,\n                        "update_seconds": list(update_seconds),\n                        "measured_update_seconds": list(update_seconds[1:])}\n            if on_update is not None:\n                on_update(progress)\n            if max_updates is not None and successful >= max_updates:\n                break\n            previous_update = now\n            if seen_batches >= planned_batches:\n                break\n    finally:\n        optimizer.zero_grad(set_to_none=True)\n    _synchronize(device)\n    if successful == 0:\n        raise FloatingPointError("Training completed no successful optimizer updates")\n    measured = update_seconds[1:]\n    return {"successful_updates": successful, "attempted_updates": attempted,\n            "overflow_retries": attempted - successful, "final_scaler_scale": float(scaler.get_scale()),\n            "examples": examples, "batches": seen_batches, "microsteps": seen_batches,\n            "mean_loss": loss_sum / max(examples, 1),\n            "elapsed_seconds": time.monotonic() - started,\n            "update_seconds": update_seconds, "measured_update_seconds": measured,\n            "seconds_per_update": float(np.mean(measured)) if measured else None,\n            "microstep_seconds": microstep_seconds,\n            "measured_microstep_seconds": microstep_seconds[min(accumulation_steps, seen_batches - 1):],\n            "seconds_per_microstep": float(np.mean(\n                microstep_seconds[min(accumulation_steps, seen_batches - 1):])),\n            "peak_vram_bytes": int(torch.cuda.max_memory_allocated(device)) if device.type == "cuda" else 0,\n            "completed_epoch": seen_batches == len(loader)}\n\n\n@torch.no_grad()\ndef predict_logits(model, loader, *, device, budget=None, max_batches=None, return_stats=False):\n    if max_batches is not None and (isinstance(max_batches, bool)\n                                    or not isinstance(max_batches, int) or max_batches < 1):\n        raise ValueError("Max batches must be a positive integer")\n    device = torch.device(device)\n    model.eval()\n    chunks = []\n    _synchronize(device)\n    started = time.monotonic()\n    for index, batch in enumerate(loader):\n        if budget is not None:\n            budget.check("classifier_inference", training=False)\n        inputs = {key: value.to(device, non_blocking=True) for key, value in batch.items()\n                  if key != "labels"}\n        with _autocast(device):\n            logits = model(**inputs)\n        values = logits.float().cpu().numpy()\n        fixed_decisions(values)\n        chunks.append(values)\n        if max_batches is not None and index + 1 >= max_batches:\n            break\n    if not chunks:\n        raise ValueError("Prediction loader is empty")\n    _synchronize(device)\n    values = np.concatenate(chunks, axis=0)\n    elapsed = time.monotonic() - started\n    if return_stats:\n        return {"logits": values, "examples": len(values), "elapsed_seconds": elapsed,\n                "seconds_per_example": elapsed / len(values)}\n    return values\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef _file_hashes(directory, excluded):\n    return {path.relative_to(directory).as_posix(): _sha256(path)\n            for path in sorted(directory.rglob("*")) if path.is_file() and path.name != excluded}\n\n\ndef _verify_files(directory, files, excluded):\n    directory = directory.resolve()\n    if not isinstance(files, dict) or not files:\n        raise ValueError("Asset manifest must contain verified files")\n    for relative, expected in files.items():\n        path = directory / relative\n        if (not isinstance(relative, str) or path.is_symlink() or Path(relative).is_absolute()\n                or not path.resolve().is_relative_to(directory) or not path.is_file()\n                or _sha256(path) != expected):\n            raise ValueError("Model asset integrity check failed")\n    actual = {path.relative_to(directory).as_posix() for path in directory.rglob("*")\n              if path.is_file() and path.name != excluded}\n    if actual != set(files):\n        raise ValueError("Model directory contains unverified files")\n\n\ndef _save_frozen_base(store, encoder, identity):\n    """Persist the clean frozen backbone before PEFT modifies its modules."""\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]\n    directory = store / key\n    manifest_path = directory / "base_manifest.json"\n    if manifest_path.exists():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest.get("identity") != identity or manifest.get("format") != FORMAT:\n            raise ValueError("Shared base identity differs from the requested pinned source")\n        # Full hashes are checked at portable reload; repeated candidate starts\n        # need not reread gigabytes of already-written frozen base weights.\n        for relative in manifest.get("files", {}):\n            if not (directory / relative).is_file():\n                raise ValueError("Shared base is incomplete")\n        return directory.resolve()\n    if directory.exists() and any(directory.iterdir()):\n        raise FileExistsError("Incomplete shared base directory; use a fresh run")\n    directory.mkdir(parents=True, exist_ok=True)\n    encoder.save_pretrained(directory, safe_serialization=True, max_shard_size="4GB")\n    _write_json(manifest_path, {"format": FORMAT, "identity": identity,\n                              "files": _file_hashes(directory, "base_manifest.json")})\n    return directory.resolve()\n\n\ndef save_model(path, model, tokenizer, spec, settings, training_scope, selected_epochs):\n    """Save a portable classifier directory and return its recipe metadata.\n\n    Qwen directories reference the run\'s shared frozen base by a relative path.\n    Copy the whole run/submission tree to preserve that relationship. No source\n    snapshot or Hugging Face cache is needed by ``load_model``.\n    """\n    directory = Path(path)\n    if directory.exists() and any(directory.iterdir()):\n        raise FileExistsError("Model save destination must be empty")\n    kind = _kind(spec)\n    if kind != model.kind:\n        raise ValueError("Model and metadata kinds differ")\n    if training_scope not in ("fit_only", "all_training", "fit", "full_training"):\n        raise ValueError("Training scope must explicitly identify fit-only or all-Training weights")\n    if isinstance(selected_epochs, bool) or not isinstance(selected_epochs, int) or selected_epochs < 1:\n        raise ValueError("Selected epochs must be a positive integer")\n    directory.mkdir(parents=True, exist_ok=True)\n    metadata = {\n        "kind": kind, "model_id": str(_get(spec, "model_id")), "base_revision": _get(spec, "revision"),\n        "max_length": int(_get(settings, f"{kind}_max_length", 1024 if kind == "qwen" else 512)),\n        "batch_size": int(_get(settings, f"{kind}_batch_size", 1 if kind == "qwen" else 8)),\n        "training_scope": training_scope, "selected_epochs": selected_epochs,\n        "loss": "BCEWithLogitsLoss_with_explicit_training_subset_pos_weight",\n        "pos_weight_rule": "clip(sqrt(N_negative/max(N_positive,1)),1,5)",\n        "decision": DECISION, "target_symptoms": list(TARGET_SYMPTOMS),\n        "judgment_suffix": JUDGMENT_SUFFIX if kind == "qwen" else None,\n    }\n    tokenizer.save_pretrained(directory / "tokenizer")\n    if kind == "qwen":\n        base = getattr(model, "_base_directory", None)\n        if base is None or not (Path(base) / "base_manifest.json").is_file():\n            raise ValueError("A Qwen checkpoint requires its separately saved clean frozen base")\n        metadata["base_relative_path"] = Path(os.path.relpath(base, directory)).as_posix()\n        metadata["base_identity"] = model._base_identity\n        metadata["base_manifest_sha256"] = _sha256(Path(base) / "base_manifest.json")\n        model.encoder.save_pretrained(directory / "adapter", safe_serialization=True)\n        torch.save({name: tensor.detach().cpu() for name, tensor in model.classifier.state_dict().items()},\n                   directory / "head.pt")\n    else:\n        metadata["dropout"] = model.dropout_probability\n        model.encoder.save_pretrained(directory / "encoder", safe_serialization=True)\n        torch.save({name: tensor.detach().cpu() for name, tensor in model.classifier.state_dict().items()},\n                   directory / "head.pt")\n    _write_json(directory / "manifest.json", {\n        "format": FORMAT, "target_symptoms": list(TARGET_SYMPTOMS), "decision": DECISION,\n        "metadata": metadata, "files": _file_hashes(directory, "manifest.json")})\n    return metadata\n\n\ndef load_model(path, device="cpu"):\n    """Integrity-check every local asset, then reload without network access."""\n    directory = Path(path)\n    if directory.is_file():\n        if directory.name != "manifest.json":\n            raise ValueError("A model checkpoint file must be its manifest.json")\n        directory = directory.parent\n    directory = directory.resolve()\n    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))\n    if (manifest.get("format") != FORMAT or manifest.get("target_symptoms") != list(TARGET_SYMPTOMS)\n            or manifest.get("decision") != DECISION or not isinstance(manifest.get("metadata"), dict)):\n        raise ValueError("Checkpoint format, labels, or fixed decision policy is invalid")\n    _verify_files(directory, manifest.get("files"), "manifest.json")\n    metadata = manifest["metadata"]\n    kind = _kind(metadata)\n    if metadata.get("target_symptoms") != list(TARGET_SYMPTOMS) or metadata.get("decision") != DECISION:\n        raise ValueError("Checkpoint metadata conflicts with the fixed label order or decision")\n    device = torch.device(device)\n    tokenizer = AutoTokenizer.from_pretrained(directory / "tokenizer", use_fast=True,\n                                               local_files_only=True, trust_remote_code=False)\n    options = dict(local_files_only=True, trust_remote_code=False)\n    if kind == "qwen":\n        from peft import PeftModel, prepare_model_for_kbit_training\n        relative = metadata.get("base_relative_path")\n        if not isinstance(relative, str) or Path(relative).is_absolute():\n            raise ValueError("Frozen base must use a portable relative path")\n        base = (directory / relative).resolve()\n        base_manifest_path = base / "base_manifest.json"\n        if _sha256(base_manifest_path) != metadata.get("base_manifest_sha256"):\n            raise ValueError("Frozen base manifest integrity check failed")\n        base_manifest = json.loads(base_manifest_path.read_text(encoding="utf-8"))\n        identity = metadata.get("base_identity")\n        if (base_manifest.get("format") != FORMAT or base_manifest.get("identity") != identity\n                or identity.get("revision") != metadata.get("base_revision")):\n            raise ValueError("Frozen base identity does not match the adapter")\n        _verify_files(base, base_manifest.get("files"), "base_manifest.json")\n        if identity.get("quantized"):\n            if device.type != "cuda":\n                raise RuntimeError("The saved NF4 Qwen model requires a CUDA device")\n            options.update(device_map={"": str(device)}, torch_dtype=_amp_dtype(device))\n        encoder = AutoModel.from_pretrained(base, **options)\n        encoder.config.use_cache = False\n        if identity.get("quantized"):\n            # Match training\'s float32 non-quantized norms/embeddings as well\n            # as its frozen 4-bit projections; omitting this changes logits.\n            encoder = prepare_model_for_kbit_training(encoder, use_gradient_checkpointing=False)\n        else:\n            encoder.to(device)\n        encoder = PeftModel.from_pretrained(encoder, directory / "adapter", is_trainable=False,\n                                            local_files_only=True)\n        model = LastTokenClassifier(encoder)\n        model._base_directory, model._base_identity = base, identity\n        model.classifier.to(device)\n    else:\n        encoder = AutoModel.from_pretrained(directory / "encoder", add_pooling_layer=False, **options)\n        model = MeanPoolClassifier(encoder, metadata["dropout"]).to(device)\n    model.classifier.load_state_dict(torch.load(directory / "head.pt", weights_only=True, map_location=device),\n                                     strict=True)\n    if tokenizer.pad_token_id is None or len(tokenizer) > model.encoder.config.vocab_size:\n        raise ValueError("Saved tokenizer and model vocabulary are incompatible")\n    if kind == "qwen" and metadata.get("judgment_suffix") != JUDGMENT_SUFFIX:\n        raise ValueError("Saved Qwen tokenization suffix differs from this model code")\n    return model.eval(), tokenizer, metadata\n\n\ndef predict_texts(model, tokenizer, texts, *, device="cpu", batch_size=16,\n                  max_length=512, budget=None):\n    loader = make_loader(tokenizer, texts, kind=model.kind, batch_size=batch_size,\n                         max_length=max_length, pin_memory=torch.device(device).type == "cuda")\n    return predict_logits(model, loader, device=device, budget=budget)\n\n\n', 'runtime.py': '"""Fresh QLoRA/control comparison with a frozen confirmation protocol."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import analysis, batching, data, trainer\nfrom .bootstrap import prepare_data\nfrom .budget import Budget, BudgetExceeded\n\nORDER = ("qwen", "control")\nFORMAT = "mission3_ver25_pointer_v1"\nEXPECTED_TRAINING_SAMPLES = 29200\nPILOT_FORMAT = "mission3_ver25_pilot_v1"\n\n\nclass _TrainingBudget:\n    def __init__(self, budget):\n        self.budget = budget\n\n    def check(self, stage, training=False):\n        return self.budget.check(stage, training=True)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef validate_settings(settings):\n    from .build_notebook import default_settings\n    defaults = default_settings()\n    result = dict(settings)\n    flexible = {"input_zip", "output_dir", "work_dir", "train_dir", "validation_dir",\n                "run_validation", "max_epochs", "time_budget_minutes", "run_mode", "source_run",\n                "finalization_reserve_seconds", "budget_safety_factor", "log_every", "allow_cpu"}\n    unknown = set(result) - set(defaults) - {"deadline_monotonic", "invocation_started_monotonic"}\n    if unknown:\n        raise ValueError("Unknown experiment settings: " + ", ".join(sorted(unknown)))\n    for key, value in defaults.items():\n        unlimited_budget = key == "time_budget_minutes" and key in result and result[key] is None\n        if key not in result or (not unlimited_budget and type(result[key]) is not type(value)):\n            raise ValueError("Missing setting or invalid type: " + key)\n        if key not in flexible and result[key] != value:\n            raise ValueError("Pinned experiment setting changed: " + key)\n    if not 1 <= result["max_epochs"] <= 3:\n        raise ValueError("Use 1-3 pilot epochs")\n    if result["run_mode"] not in ("pilot", "full"):\n        raise ValueError("run_mode must be pilot or full")\n    if result["run_mode"] == "full" and not result["source_run"].strip():\n        raise ValueError("Full training requires a completed pilot source_run")\n    if result["run_mode"] == "pilot" and result["source_run"]:\n        raise ValueError("Set source_run only for a separate full run")\n    minutes = result["time_budget_minutes"]\n    if ((minutes is not None and (minutes <= 0\n                                 or result["finalization_reserve_seconds"] >= 60 * minutes))\n            or not 1200 <= result["finalization_reserve_seconds"] <= 1800\n            or not math.isfinite(result["budget_safety_factor"])\n            or result["budget_safety_factor"] < 1.25 or result["log_every"] < 1):\n        raise ValueError("Invalid time budget or reserve")\n    for key in ("deadline_monotonic", "invocation_started_monotonic"):\n        if key not in result:\n            continue\n        timestamp = result[key]\n        if key == "deadline_monotonic" and minutes is None and timestamp is None:\n            continue\n        if (isinstance(timestamp, bool) or not isinstance(timestamp, (int, float))\n                or not math.isfinite(timestamp) or timestamp < 0):\n            raise ValueError("Invalid invocation timestamp: " + key)\n    return result\n\n\ndef dataset_digest(texts, targets):\n    return data.dataset_digest(texts, targets)\n\n\ndef _hash(path, budget=None):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            if budget:\n                budget.check("artifact hash")\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _save_npz(path, **arrays):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    temporary.replace(path)\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _device(settings):\n    import torch\n    if torch.cuda.is_available():\n        return "cuda"\n    if not settings["allow_cpu"]:\n        raise RuntimeError("CUDA GPU required for the QLoRA experiment")\n    return "cpu"\n\n\ndef resolve_specs(cache_dir, budget, source_specs=None):\n    """Resolve mutable references once, then use immutable snapshots everywhere."""\n    from huggingface_hub import HfApi, snapshot_download\n    specs = (read_json(Path(__file__).with_name("model_specs.json")) if source_specs is None\n             else {kind: dict(source_specs[kind]) for kind in ORDER})\n    for name in ORDER:\n        budget.check("resolve pretrained revision", training=True)\n        spec = specs[name]\n        revision = spec["revision"]\n        if not re.fullmatch(r"[0-9a-f]{40}", revision):\n            revision = HfApi().model_info(spec["model_id"], revision=revision).sha\n        if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):\n            raise ValueError("Pretrained model revision is not immutable")\n        snapshot = snapshot_download(spec["model_id"], revision=revision, cache_dir=str(cache_dir),\n            allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja", "LICENSE*", "README.md"],\n            ignore_patterns=["original/*", "onnx/*", "openvino/*"])\n        spec.update(kind=name, revision=revision, snapshot_path=str(snapshot))\n        budget.check("pretrained download", training=True)\n    return specs\n\n\ndef _loader(kind, tokenizer, texts, targets, indices, settings, *, shuffle=False, device="cpu"):\n    return trainer.make_loader(tokenizer, texts, targets=targets, indices=indices, kind=kind,\n        batch_size=settings[kind + "_batch_size"], max_length=settings[kind + "_max_length"],\n        shuffle=shuffle, seed=settings["seed"], pin_memory=device == "cuda")\n\n\ndef _optimizer(kind, model, loader, horizon, settings, device):\n    optimizer = trainer.build_optimizer(model, kind, settings)\n    updates = math.ceil(len(loader) / settings[kind + "_gradient_accumulation"]) * horizon\n    scheduler = trainer.build_scheduler(optimizer, updates, settings["warmup_ratio"])\n    return optimizer, scheduler, trainer.make_scaler(device)\n\n\ndef _progress(name, every):\n    def report(progress):\n        if progress["successful_updates"] % every == 0:\n            print(f"{name}: {progress[\'examples\']} rows; {progress[\'elapsed_seconds\'] / 60:.1f} min", flush=True)\n    return report\n\n\ndef timing_estimates(probes, fit_count, full_count, development_count, confirmation_count, settings):\n    estimates = {}\n    for kind in ORDER:\n        probe = probes[kind]\n        step, predict = probe["seconds_per_example"], probe["seconds_per_prediction"]\n        if not all(math.isfinite(x) and x > 0 for x in (step, predict)):\n            raise ValueError("Pilot timings must be finite and positive")\n        batch = settings[kind + "_batch_size"]\n        estimates[kind] = {"fit_epoch_seconds": step * math.ceil(fit_count / batch) * batch,\n            "development_seconds": predict * development_count,\n            "confirmation_seconds": predict * confirmation_count,\n            "full_epoch_seconds": step * math.ceil(full_count / batch) * batch,\n            "setup_seconds": max(30., probe.get("setup_seconds", 30.))}\n    return estimates\n\n\ndef plan_epochs(probes, fit_count, full_count, development_count, confirmation_count, settings, budget):\n    """Admit only the requested small comparison, never a full-data refit."""\n    estimates = timing_estimates(probes, fit_count, full_count, development_count, confirmation_count, settings)\n    for epochs in range(settings["max_epochs"], 0, -1):\n        comparison = sum((p["fit_epoch_seconds"] + p["development_seconds"]) * epochs\n            + 2 * p["setup_seconds"] + 60 * epochs for p in estimates.values())\n        if budget.can_start(comparison, training=True):\n            return {"common_epochs": epochs, "estimates": estimates,\n                "full_refit_planned": False, "confirmation_planned": False,\n                "estimated_work_seconds": comparison, "comparison_seconds": comparison,\n                "fitted_samples": fit_count, "remaining_seconds": budget.remaining(),\n                "safety_factor": budget.safety_factor,\n                "finalization_reserve_seconds": budget.finalization_reserve_s}\n    raise BudgetExceeded("one complete subset epoch and development evaluation do not fit")\n\n\ndef estimate_full_run(estimates, selection, settings, *, records=None, fit_count=None, full_count=None,\n                      preparation_seconds=0.):\n    selected = selection["components"]\n    refit = 0.\n    for kind in selected:\n        epoch_seconds = estimates[kind]["full_epoch_seconds"]\n        if records is not None:\n            epoch_seconds = max(epoch_seconds, records[kind]["actual_epoch_seconds"] * full_count / fit_count)\n        refit += epoch_seconds * selection["selected_epochs"][kind] + estimates[kind]["setup_seconds"]\n    confirmation = sum(p["confirmation_seconds"] + p["setup_seconds"] for p in estimates.values())\n    # Use development-size inference as a conservative proxy without opening official Validation.\n    validation = sum(estimates[kind]["development_seconds"] for kind in selected)\n    raw = refit + confirmation + validation + 120 * len(selected)\n    if not math.isfinite(preparation_seconds) or preparation_seconds < 0:\n        raise ValueError("Preparation estimate must be finite and nonnegative")\n    return {"selected_candidate": selection["selected_candidate"], "selected_epochs": selection["selected_epochs"],\n        "refit_seconds": refit, "confirmation_seconds": confirmation,\n        "validation_proxy_seconds": validation, "estimated_work_seconds": raw,\n        "estimated_preparation_seconds": preparation_seconds,\n        "recommended_budget_minutes": math.ceil(((raw + preparation_seconds) * settings["budget_safety_factor"]\n            + settings["finalization_reserve_seconds"]) / 60),\n        "safety_factor": settings["budget_safety_factor"],\n        "notes": "Estimate only; includes measured preparation/batch-search time, safety margin and export reserve. "\n                 "Future source verification, installation/download and GPU speeds can differ. "\n                 "Batch search repeats on the future GPU. No full run starts automatically."}\n\n\ndef choose_candidate(scores, records, min_macro_gain=.01):\n    # Stable tie order prefers the simpler single model, with control first.\n    selected = max(("control", "qwen", "ensemble"), key=lambda name: scores[name]["macro_f1"])\n    gain = scores[selected]["macro_f1"] - scores["control"]["macro_f1"]\n    components = ["control", "qwen"] if selected == "ensemble" else [selected]\n    return {"selected_candidate": selected, "components": components,\n        "selected_epochs": {name: records[name]["selected_epochs"] for name in components},\n        "selection_data": "Training development only", "confirmation_can_change_selection": False,\n        "macro_f1_delta": gain, "improved": gain > 0,\n        "expansion_recommended": gain >= min_macro_gain,\n        "expansion_macro_gain": min_macro_gain,\n        "rule": "highest development Macro F1; ties control then qwen then fixed mean; no nausea veto",\n        "historical_best_comparison": "not evaluated; this control is a fresh single KLUE model"}\n\n\ndef _fit_candidate(kind, spec, texts, targets, fit, development, settings, plan, device, budget, root):\n    budget.check("candidate load", training=True)\n    model, tokenizer = trainer.create_model(spec, settings, device)\n    loader = _loader(kind, tokenizer, texts, targets, fit, settings, shuffle=True, device=device)\n    validation = _loader(kind, tokenizer, texts, None, development, settings, device=device)\n    optimizer, scheduler, scaler = _optimizer(kind, model, loader, plan["common_epochs"], settings, device)\n    weights = trainer.positive_weights(targets[fit])\n    best, best_logits, history = None, None, []\n    for epoch in range(1, plan["common_epochs"] + 1):\n        print(f"{kind}: fit epoch {epoch}/{plan[\'common_epochs\']}", flush=True)\n        stats = trainer.train_epoch(model, loader, optimizer, device=device, pos_weight=weights,\n            accumulation_steps=settings[kind + "_gradient_accumulation"], scheduler=scheduler,\n            scaler=scaler, budget=budget, on_update=_progress(kind, settings["log_every"]))\n        if not stats["completed_epoch"]:\n            raise RuntimeError("Candidate epoch did not finish")\n        evaluation_started = time.monotonic()\n        logits = trainer.predict_logits(model, validation, device=device, budget=_TrainingBudget(budget))\n        evaluation_seconds = time.monotonic() - evaluation_started\n        score = analysis.metrics(targets[development], logits)\n        record = {"candidate": kind, "macro_f1": score["macro_f1"], "selected_epochs": epoch,\n            "train_seconds": stats["elapsed_seconds"], "peak_vram_bytes": stats.get("peak_vram_bytes"),\n            "evaluation_seconds": evaluation_seconds, "overflow_retries": stats.get("overflow_retries", 0),\n            "fitted_samples": len(fit), "positive_weights": weights.tolist()}\n        history.append({**record, "metrics": score})\n        if best is None or score["macro_f1"] > best["macro_f1"]:\n            best, best_logits = record, logits.copy()\n            budget.check("save fit checkpoint")\n            trainer.save_model(root / ("fit_" + kind) / ("epoch_" + str(epoch)),\n                               model, tokenizer, spec, settings, "fit_only", epoch)\n            data.write_json(root / (kind + "_development_metrics.json"), score)\n            _save_npz(root / (kind + "_development_predictions.npz"), indices=development,\n                targets=targets[development], logits=best_logits, predictions=trainer.fixed_decisions(best_logits))\n        data.write_json(root / (kind + "_history.json"), history)\n        print(f"{kind}: development Macro F1={score[\'macro_f1\']:.6f}", flush=True)\n    best["actual_epoch_seconds"] = max(row["train_seconds"] for row in history)\n    del model, tokenizer, loader, validation, optimizer, scheduler, scaler\n    _release()\n    return best, best_logits\n\n\ndef _compare(root, stage, targets, indices, logits, budget):\n    logits = dict(logits)\n    logits["ensemble"] = analysis.ensemble_logits(logits["control"], logits["qwen"])\n    scores = {name: analysis.metrics(targets[indices], values) for name, values in logits.items()}\n    changes = {name: analysis.paired_error_changes(targets[indices], logits["control"], logits[name])\n               for name in ("qwen", "ensemble")}\n    intervals = {}\n    for name in ("qwen", "ensemble"):\n        budget.check("paired call bootstrap", training=True)\n        intervals[name] = analysis.paired_bootstrap_ci(targets[indices], logits["control"], logits[name])\n    data.write_json(root / (stage + "_comparison.json"), {"metrics": scores,\n        "paired_error_changes": changes, "paired_bootstrap": intervals,\n        "bootstrap_unit": "call", "decision": "fixed sigmoid >= 0.5",\n        "ensemble": "fixed 1:1 probability mean, mathematical tie handled by logit sum sign"})\n    _save_npz(root / (stage + "_predictions.npz"), indices=indices, targets=targets[indices],\n              **{name + "_logits": values for name, values in logits.items()})\n    return scores\n\n\ndef _confirmation(root, texts, targets, indices, records, settings, device, budget, *, source_root=None):\n    logits = {}\n    source_root = root if source_root is None else source_root\n    for kind in ORDER:\n        budget.check("confirmation reload", training=True)\n        model, tokenizer, _ = trainer.load_model(source_root / ("fit_" + kind)\n            / ("epoch_" + str(records[kind]["selected_epochs"])), device)\n        loader = _loader(kind, tokenizer, texts, None, indices, settings, device=device)\n        logits[kind] = trainer.predict_logits(model, loader, device=device, budget=_TrainingBudget(budget))\n        del model, tokenizer, loader\n        _release()\n    return _compare(root, "confirmation", targets, indices, logits, budget)\n\n\ndef _refit(kind, spec, epochs, texts, targets, settings, plan, device, budget, root):\n    budget.check("full Training refit load", training=True)\n    model, tokenizer = trainer.create_model(spec, settings, device)\n    loader = _loader(kind, tokenizer, texts, targets, None, settings, shuffle=True, device=device)\n    optimizer, scheduler, scaler = _optimizer(kind, model, loader, plan["common_epochs"], settings, device)\n    weights = trainer.positive_weights(targets)\n    history = []\n    for epoch in range(1, epochs + 1):\n        print(f"{kind}: full Training epoch {epoch}/{epochs}", flush=True)\n        stats = trainer.train_epoch(model, loader, optimizer, device=device, pos_weight=weights,\n            accumulation_steps=settings[kind + "_gradient_accumulation"], scheduler=scheduler,\n            scaler=scaler, budget=budget, on_update=_progress("refit " + kind, settings["log_every"]))\n        if not stats["completed_epoch"] or stats["examples"] != len(texts):\n            raise RuntimeError("Full Training refit did not cover all Training rows")\n        history.append({"epoch": epoch, "examples": stats["examples"],\n            "seconds": stats["elapsed_seconds"], "peak_vram_bytes": stats.get("peak_vram_bytes"),\n            "overflow_retries": stats.get("overflow_retries", 0)})\n        data.write_json(root / (kind + "_final_history.json"), history)\n    directory = root / ("final_" + kind)\n    budget.check("export final component")\n    trainer.save_model(directory, model, tokenizer, spec, settings, "all_training", epochs)\n    sample = [texts[i] for i in np.linspace(0, len(texts) - 1, min(16, len(texts)), dtype=int)]\n    expected = trainer.predict_logits(model, _loader(kind, tokenizer, sample, None, None, settings),\n                                      device=device, budget=budget)\n    del model, tokenizer, loader, optimizer, scheduler, scaler\n    _release()\n    budget.check("reload final component")\n    model, tokenizer, metadata = trainer.load_model(directory, device)\n    if metadata["training_scope"] != "all_training":\n        raise ValueError("Final component is not a full Training model")\n    actual = trainer.predict_logits(model, _loader(kind, tokenizer, sample, None, None, settings),\n                                    device=device, budget=budget)\n    verified = (np.allclose(expected, actual, rtol=1e-4, atol=1e-4)\n                and np.array_equal(trainer.fixed_decisions(expected), trainer.fixed_decisions(actual)))\n    if not verified:\n        raise ValueError("Locally reloaded checkpoint predictions differ")\n    data.write_json(root / (kind + "_portable_verification.json"), {"samples": len(sample),\n        "decisions_equal": True, "local_files_only": True,\n        "max_abs_logit_difference": float(np.max(abs(expected - actual))),\n        "positive_weights": weights.tolist(), "fitted_samples": len(texts),\n        "scheduler_horizon_epochs": plan["common_epochs"]})\n    del model, tokenizer\n    _release()\n    return {"kind": kind, "model_directory": directory.name, "selected_epochs": epochs}\n\n\ndef _sources(root, runtime_sources):\n    sources = runtime_sources or {p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.iterdir()\n        if p.suffix in {".py", ".txt", ".json", ".md"} and not p.name.startswith("test_")}\n    submission = root / "submission"\n    submission.mkdir(exist_ok=False)\n    for name, source in sources.items():\n        if Path(name).name != name:\n            raise ValueError("Unsafe source bundle name")\n        (submission / name).write_text(source, encoding="utf-8")\n    packages = ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "numpy", "scikit-learn")\n    versions = {}\n    for package in packages:\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = "not-installed"\n    data.write_json(root / "environment.json", versions)\n\n\ndef _diagnostic(root, status, stage, started, records=None):\n    saved_fit = sorted(p.parent.relative_to(root).as_posix()\n                       for p in root.glob("fit_*/epoch_*/manifest.json"))\n    if saved_fit and status == "budget_not_started":\n        status = "fit_only_budget"\n    result = {"version": "ver25", "status": status, "stage": stage,\n        "training_scope": "fit_only" if records or saved_fit else "none", "submission_ready": False,\n        "elapsed_worker_seconds": time.monotonic() - started,\n        "completed_fit_candidates": list(records or {}), "saved_fit_checkpoints": saved_fit,\n        "validation_evaluated": (root / "validation_metrics.json").is_file()}\n    data.write_json(root / "runtime.json", result)\n    data.write_json(root / "diagnostic_complete.json", result)\n    return root\n\n\ndef _verify_pilot_exports(root, texts, development, records, logits, settings, device, budget):\n    verification = {}\n    indices = development[:16]\n    for kind in ORDER:\n        budget.check("reload pilot checkpoint")\n        model, tokenizer, metadata = trainer.load_model(root / ("fit_" + kind)\n            / ("epoch_" + str(records[kind]["selected_epochs"])), device)\n        if metadata["training_scope"] != "fit_only":\n            raise ValueError("Pilot component has an unexpected training scope")\n        actual = trainer.predict_logits(model,\n            _loader(kind, tokenizer, texts, None, indices, settings, device=device),\n            device=device, budget=budget)\n        expected = logits[kind][:len(indices)]\n        if (not np.allclose(expected, actual, rtol=1e-4, atol=1e-4)\n                or not np.array_equal(trainer.fixed_decisions(expected), trainer.fixed_decisions(actual))):\n            raise ValueError("Locally reloaded pilot predictions differ")\n        verification[kind] = {"samples": len(indices), "decisions_equal": True,\n            "local_files_only": True, "max_abs_logit_difference": float(np.max(abs(expected - actual)))}\n        del model, tokenizer\n        _release()\n    data.write_json(root / "pilot_portable_verification.json", verification)\n\n\ndef _load_pilot_source(source_run, texts, targets, fit, development, confirmation, subset,\n                       split_report, subset_report, settings, budget):\n    source = Path(source_run).expanduser().resolve()\n    marker_path = source / "pilot_complete.json"\n    if not marker_path.is_file():\n        raise ValueError("source_run must contain a completed pilot_complete.json; incomplete runs cannot start full training")\n    marker = read_json(marker_path)\n    if (marker.get("format") != PILOT_FORMAT or marker.get("status") != "pilot_complete"\n            or marker.get("training_scope") != "training_subset" or marker.get("submission_ready") is not False\n            or marker.get("confirmation_evaluated") is not False or marker.get("validation_evaluated") is not False):\n        raise ValueError("Invalid completed pilot source")\n    hashes = marker.get("hashes", {})\n    required = {"pilot_recipe.json", "final_selection.json", "model_specs.json", "settings.json",\n        "training_data.json", "split.npz", "split_manifest.json", "subset.npz", "subset_manifest.json",\n        "pilot_portable_verification.json", "development_comparison.json"}\n    if not required.issubset(hashes):\n        raise ValueError("Pilot completion marker is missing required artifact hashes")\n    for name, expected in hashes.items():\n        relative = Path(name)\n        path = source / relative\n        if (relative.is_absolute() or ".." in relative.parts or path.is_symlink()\n                or not path.resolve().is_relative_to(source) or not path.is_file()\n                or _hash(path, budget) != expected):\n            raise ValueError("Pilot source artifact is missing or changed: " + name)\n    previous = validate_settings(read_json(source / "settings.json"))\n    if previous["run_mode"] != "pilot":\n        raise ValueError("source_run must be a pilot run")\n    recipe = read_json(source / "pilot_recipe.json")\n    selection = read_json(source / "final_selection.json")\n    specs = read_json(source / "model_specs.json")\n    if (recipe.get("format") != PILOT_FORMAT or recipe.get("target_symptoms") != list(data.TARGET_SYMPTOMS)\n            or recipe.get("training_dataset_digest") != dataset_digest(texts, targets)\n            or recipe.get("selection") != selection\n            or recipe.get("common_epochs") != selection.get("common_epochs")\n            or not 1 <= recipe.get("common_epochs", 0) <= previous["max_epochs"]):\n        raise ValueError("Pilot recipe or Training data does not match this full run")\n    for name, current in (("split_manifest.json", split_report), ("subset_manifest.json", subset_report)):\n        if read_json(source / name) != current:\n            raise ValueError("Pilot split or subset does not match current Training data")\n    with np.load(source / "split.npz", allow_pickle=False) as saved:\n        if any(not np.array_equal(saved[key], values)\n               for key, values in (("fit", fit), ("development", development), ("confirmation", confirmation))):\n            raise ValueError("Pilot split indices changed")\n    with np.load(source / "subset.npz", allow_pickle=False) as saved:\n        if not np.array_equal(saved["indices"], subset):\n            raise ValueError("Pilot subset indices changed")\n    records = recipe.get("records", {})\n    scores = read_json(source / "development_comparison.json")["metrics"]\n    expected_selection = choose_candidate(scores, records, previous["min_macro_gain"])\n    if any(selection.get(key) != value for key, value in expected_selection.items()):\n        raise ValueError("Pilot selection differs from the frozen development decision")\n    if selection.get("training_dataset_digest") != recipe["training_dataset_digest"]:\n        raise ValueError("Pilot selection data digest differs")\n    for kind in ORDER:\n        epoch = records[kind]["selected_epochs"]\n        if type(epoch) is not int or not 1 <= epoch <= recipe["common_epochs"]:\n            raise ValueError("Invalid selected pilot epoch")\n        if (f"fit_{kind}/epoch_{epoch}/manifest.json" not in hashes\n                or not re.fullmatch(r"[0-9a-f]{40}", specs[kind]["revision"])):\n            raise ValueError("Pilot source lacks a selected checkpoint or pinned pretrained revision")\n    # validate_settings pins every model/data hyperparameter in both runs. Only\n    # the new run\'s paths, timing budget, hardware, and measured batches change.\n    return source, recipe, specs\n\n\ndef run(settings, runtime_sources=None):\n    settings = validate_settings(settings)\n    started = time.monotonic()\n    minutes = settings["time_budget_minutes"]\n    deadline = (None if minutes is None else\n                min(float(settings.get("deadline_monotonic", started + minutes * 60)), started + minutes * 60))\n    invocation_started = settings.get("invocation_started_monotonic",\n                                      started if deadline is None else deadline - minutes * 60)\n    budget = Budget(deadline, settings["finalization_reserve_seconds"], settings["budget_safety_factor"])\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    root = Path(settings["output_dir"]).expanduser().resolve() / stamp\n    root.mkdir(parents=True, exist_ok=False)\n    data.write_json(root / "settings.json", settings)\n    settings = {**settings, "base_store_dir": str(root / "shared_base")}\n    stage, records = "prepare", {}\n    data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n    try:\n        _sources(root, runtime_sources)\n        device = _device(settings)\n        import torch\n        hardware = {"device": device, "torch_version": torch.__version__,\n                    "cuda_version": torch.version.cuda}\n        if device == "cuda":\n            properties = torch.cuda.get_device_properties(0)\n            hardware.update(gpu_name=properties.name, vram_bytes=properties.total_memory,\n                            bf16_supported=torch.cuda.is_bf16_supported())\n        data.write_json(root / "hardware.json", hardware)\n        budget.check(stage, training=True)\n        train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n            train_dir=settings["train_dir"], validation_dir=settings["validation_dir"])\n        _, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        if len(texts) != EXPECTED_TRAINING_SAMPLES:\n            raise ValueError("Training must contain all 29,200 provided calls before model downloads")\n        fit, development, confirmation, split_report = data.three_way_split(texts, targets, settings["seed"])\n        subset, subset_report = data.sample_fit_subset(texts, targets, fit,\n            sample_count=settings["pilot_samples"], seed=settings["seed"])\n        _save_npz(root / "split.npz", fit=fit, development=development, confirmation=confirmation)\n        data.write_json(root / "split_manifest.json", split_report)\n        _save_npz(root / "subset.npz", indices=subset)\n        data.write_json(root / "subset_manifest.json", subset_report)\n        digest = dataset_digest(texts, targets)\n        data.write_json(root / "training_data.json", {"samples": len(texts), "fit_samples": len(fit),\n            "pilot_samples": len(subset), "requested_pilot_samples": settings["pilot_samples"],\n            "development_samples": len(development), "confirmation_samples": len(confirmation),\n            "digest": digest, "labels_changed": 0, "samples_removed": 0,\n            "training_count_matches_expected_29200": len(texts) == EXPECTED_TRAINING_SAMPLES})\n        data.write_json(root / "training_diagnostics.json", data.training_diagnostics(texts, targets))\n        source, source_recipe, source_specs = None, None, None\n        if settings["run_mode"] == "full":\n            stage = "validate_pilot_source"\n            source, source_recipe, source_specs = _load_pilot_source(settings["source_run"], texts, targets,\n                fit, development, confirmation, subset, split_report, subset_report, settings, budget)\n            data.write_json(root / "source_pilot.json", {"source_run": str(source),\n                "completion_sha256": _hash(source / "pilot_complete.json", budget),\n                "training_dataset_digest": digest, "selection_reused_without_search": True})\n        stage = "pretrained_download"\n        specs = resolve_specs(Path(settings["work_dir"]) / "hf_cache", budget, source_specs=source_specs)\n        # Do not put machine-specific HF cache paths into the portable recipe.\n        data.write_json(root / "model_specs.json", {k: {a: b for a, b in v.items() if a != "snapshot_path"}\n                                                  for k, v in specs.items()})\n        for kind, spec in specs.items():\n            notices = root / "licenses" / kind\n            notices.mkdir(parents=True)\n            copied = []\n            snapshot = Path(spec["snapshot_path"])\n            for notice_path in sorted(snapshot.iterdir()):\n                if notice_path.is_file() and (notice_path.name.startswith("LICENSE") or notice_path.name == "README.md"):\n                    shutil.copyfile(notice_path, notices / notice_path.name)\n                    copied.append(notice_path.name)\n            data.write_json(notices / "attribution.json", {"model_id": spec["model_id"],\n                "revision": spec["revision"], "source": "https://huggingface.co/" + spec["model_id"],\n                "license_note": spec.get("license_note"), "upstream_files": copied,\n                "modification": "9-output classifier trained on provided Training; Qwen NF4 and LoRA adaptation"})\n        probes = {}\n        for kind in ORDER:\n            stage = kind + "_batch_benchmark"\n            data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n            print(f"Measuring {kind} batch throughput; benchmark weights are discarded.", flush=True)\n            settings, probes[kind] = batching.autotune(kind, specs[kind], texts, targets,\n                subset, development, settings, device, budget, root)\n            data.write_json(root / "batch_selection.json", probes)\n        data.write_json(root / "effective_settings.json", settings)\n        preparation_seconds = max(0., time.monotonic() - invocation_started)\n        stage = "budget_admission"\n        if settings["run_mode"] == "pilot":\n            plan = plan_epochs(probes, len(subset), len(texts), len(development), len(confirmation), settings, budget)\n        else:\n            records, selection = source_recipe["records"], source_recipe["selection"]\n            estimates = timing_estimates(probes, len(subset), len(texts), len(development), len(confirmation), settings)\n            full_estimate = estimate_full_run(estimates, selection, settings,\n                preparation_seconds=preparation_seconds)\n            data.write_json(root / "full_run_estimate.json", full_estimate)\n            plan = {"common_epochs": source_recipe["common_epochs"], "estimates": estimates,\n                "full_refit_planned": True, "confirmation_planned": True,\n                "estimated_work_seconds": full_estimate["estimated_work_seconds"],\n                "remaining_seconds": budget.remaining(), "fitted_samples": len(texts)}\n            if not budget.can_start(full_estimate["estimated_work_seconds"], training=True):\n                data.write_json(root / "budget_plan.json", {**plan, "admitted": False})\n                return _diagnostic(root, "budget_not_started", "full_training_admission", started)\n        data.write_json(root / "budget_plan.json", plan)\n        if settings["run_mode"] == "pilot":\n            logits = {}\n            for kind in ORDER:\n                stage = kind + "_fit"\n                data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n                records[kind], logits[kind] = _fit_candidate(kind, specs[kind], texts, targets, subset, development,\n                    settings, plan, device, budget, root)\n            stage = "development_selection"\n            scores = _compare(root, "development", targets, development, logits, budget)\n            with (root / "candidate_summary.csv").open("w", encoding="utf-8-sig", newline="") as handle:\n                writer = csv.writer(handle)\n                writer.writerow(["candidate", "macro_f1", "nausea_f1", "selected_epochs"])\n                for name in ("control", "qwen", "ensemble"):\n                    epochs = records[name]["selected_epochs"] if name in records else json.dumps(\n                        {kind: records[kind]["selected_epochs"] for kind in ORDER})\n                    writer.writerow([name, scores[name]["macro_f1"], scores[name]["nausea_f1"], epochs])\n            selection = choose_candidate(scores, records, settings["min_macro_gain"])\n            selection.update(common_epochs=plan["common_epochs"], training_dataset_digest=digest)\n            data.write_json(root / "final_selection.json", selection)\n            data.write_json(root / "training_metrics.json", scores[selection["selected_candidate"]])\n            data.write_json(root / "pilot_recipe.json", {"format": PILOT_FORMAT,\n                "target_symptoms": list(data.TARGET_SYMPTOMS), "training_dataset_digest": digest,\n                "common_epochs": plan["common_epochs"], "records": records, "selection": selection})\n            data.write_json(root / "full_run_estimate.json", estimate_full_run(plan["estimates"], selection, settings,\n                records=records, fit_count=len(subset), full_count=len(texts), preparation_seconds=preparation_seconds))\n            stage = "verify_pilot_export"\n            _verify_pilot_exports(root, texts, development, records, logits, settings, device, budget)\n            report = {"version": "ver25", "format": PILOT_FORMAT, "run_mode": "pilot",\n                "status": "pilot_complete", "training_scope": "training_subset", "submission_ready": False,\n                "fitted_samples": len(subset), "training_samples": len(texts), "device": device,\n                "confirmation_evaluated": False, "validation_evaluated": False,\n                "elapsed_worker_seconds": time.monotonic() - started,\n                "elapsed_invocation_seconds": max(0., time.monotonic() - invocation_started)}\n            data.write_json(root / "runtime.json", report)\n            hashes = {p.relative_to(root).as_posix(): _hash(p, budget)\n                      for p in sorted(root.rglob("*")) if p.is_file()}\n            budget.check("publish pilot completion marker")\n            data.write_json(root / "pilot_complete.json", {**report, "hashes": hashes})\n            print("Pilot complete: results saved. Full training requires a separate explicit full run.", flush=True)\n            return root\n\n        data.write_json(root / "final_selection.json", selection)\n        data.write_json(root / "pilot_recipe.json", source_recipe)\n        # Selection is written BEFORE the reserved confirmation labels are evaluated.\n        stage = "frozen_confirmation"\n        confirmed = _confirmation(root, texts, targets, confirmation, records, settings, device, budget, source_root=source)\n        data.write_json(root / "confirmation_protocol.json", {"selection_frozen_before_evaluation": True,\n            "selection_changed": False, "comparison_count": 1,\n            "selected_candidate": selection["selected_candidate"],\n            "selected_macro_delta": confirmed[selection["selected_candidate"]]["macro_f1"] - confirmed["control"]["macro_f1"],\n            "limitation": "Training has historical experiment exposure; this is not a new external test set."})\n        stage = "full_training_admission"\n        expected = sum(plan["estimates"][kind]["full_epoch_seconds"] * selection["selected_epochs"][kind]\n            + plan["estimates"][kind]["setup_seconds"] for kind in selection["components"])\n        if not budget.can_start(expected, training=True):\n            return _diagnostic(root, "budget_not_started", stage, started)\n        components = []\n        for kind in selection["components"]:\n            stage = kind + "_full_training_refit"\n            data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n            components.append(_refit(kind, specs[kind], selection["selected_epochs"][kind], texts, targets,\n                                      settings, plan, device, budget, root))\n        manifest = {"format": FORMAT, "version": "ver25", "training_scope": "all_training",\n            "fitted_samples": len(texts), "training_dataset_digest": digest,\n            "target_symptoms": list(data.TARGET_SYMPTOMS), "components": components,\n            "combination": "fixed_probability_mean" if len(components) == 2 else "single",\n            "decision": "independent sigmoid >= 0.5", "selection": selection}\n        checkpoint = root / "mission3_ver25_model.ckpt"\n        data.write_json(checkpoint, manifest)\n        stage = "final_validation"\n        if settings["run_validation"]:\n            budget.check(stage)\n            paths, validation_texts, validation_targets = data.load_dataset(validation_dir, include_targets=True)\n            final_logits = _predict_components(root, manifest, validation_texts, device, budget)\n            predictions = trainer.fixed_decisions(final_logits)\n            data.write_predictions(paths, predictions, root / "mission3.csv")\n            data.write_json(root / "validation_metrics.json", analysis.metrics(validation_targets, final_logits))\n            _save_npz(root / "validation_predictions.npz", logits=final_logits,\n                      targets=validation_targets, predictions=predictions)\n        stage = "finalize"\n        status = "complete" if selection["improved"] else "complete_not_improved"\n        report = {"version": "ver25", "run_mode": "full", "status": status, "training_scope": "all_training",\n            "fitted_samples": len(texts), "validation_evaluated": settings["run_validation"],\n            "device": device, "elapsed_worker_seconds": time.monotonic() - started,\n            "elapsed_invocation_seconds": max(0., time.monotonic() - invocation_started),\n            "submission_ready": settings["run_validation"]}\n        data.write_json(root / "runtime.json", report)\n        hashes = {p.relative_to(root).as_posix(): _hash(p, budget) for p in sorted(root.rglob("*")) if p.is_file()}\n        budget.check("publish completion marker")\n        data.write_json(root / "run_complete.json", {**report, "hashes": hashes})\n        return root\n    except BudgetExceeded:\n        # Completed best-epoch fit checkpoints remain valid diagnostics. A\n        # partial all-data refit never changes their scope or becomes complete.\n        local_records = records if settings["run_mode"] == "pilot" else {}\n        status = "fit_only_budget" if local_records else "budget_not_started"\n        return _diagnostic(root, status, stage, started, local_records)\n    except Exception as error:\n        data.write_json(root / "runtime.json", {"status": "failed", "stage": stage,\n            "error_type": type(error).__name__, "elapsed_worker_seconds": time.monotonic() - started,\n            "submission_ready": False})\n        raise\n\n\ndef _validated_manifest(checkpoint):\n    checkpoint = Path(checkpoint).resolve()\n    manifest = read_json(checkpoint)\n    if (manifest.get("format") != FORMAT or manifest.get("training_scope") != "all_training"\n            or manifest.get("target_symptoms") != list(data.TARGET_SYMPTOMS)):\n        raise ValueError("Unrecognized or fit-only ver25 checkpoint")\n    components = manifest.get("components", [])\n    names = [item.get("kind") for item in components]\n    if names not in (["control"], ["qwen"], ["control", "qwen"]):\n        raise ValueError("Invalid model components")\n    expected = "single" if len(names) == 1 else "fixed_probability_mean"\n    if manifest.get("combination") != expected:\n        raise ValueError("Invalid fixed model combination")\n    for item in components:\n        if item.get("model_directory") != "final_" + item["kind"]:\n            raise ValueError("Invalid component path")\n        path = checkpoint.parent / item["model_directory"]\n        if path.is_symlink() or not path.resolve().is_relative_to(checkpoint.parent):\n            raise ValueError("Model component escapes run directory")\n    return checkpoint.parent, manifest\n\n\ndef _predict_components(root, manifest, texts, device, budget=None):\n    logits = []\n    for item in manifest["components"]:\n        if budget:\n            budget.check("offline component load")\n        model, tokenizer, metadata = trainer.load_model(root / item["model_directory"], device)\n        if metadata["training_scope"] != "all_training":\n            raise ValueError("Inference component was not refit on full Training")\n        loader = trainer.make_loader(tokenizer, texts, kind=item["kind"],\n            batch_size=metadata.get("batch_size", 1), max_length=metadata["max_length"])\n        logits.append(trainer.predict_logits(model, loader, device=device, budget=budget))\n        del model, tokenizer, loader\n        _release()\n    return logits[0] if len(logits) == 1 else analysis.ensemble_logits(*logits)\n\n\ndef inference(checkpoint, label_dir, output):\n    import torch\n    root, manifest = _validated_manifest(checkpoint)\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    logits = _predict_components(root, manifest, texts, "cuda" if torch.cuda.is_available() else "cpu")\n    data.write_predictions(paths, trainer.fixed_decisions(logits), output)\n    return Path(output)\n\n\ndef inference_main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", default="", help="Interface compatibility only; audio is unused")\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", "--checkpoint", dest="checkpoint", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args(argv)\n    inference(args.checkpoint, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    inference_main()\n', 'inference.py': '"""One-command inference entry point for a portable ver25 result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import inference_main as main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver25_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").inference_main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Launch the bounded QLoRA representation comparison in a fresh interpreter."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport re\nimport sys\nimport traceback\n\nSOURCE_FILES = (\n    "__init__.py", "data.py", "analysis.py", "bootstrap.py", "budget.py", "batching.py", "trainer.py", "runtime.py",\n    "inference.py", "run_notebook.py", "build_notebook.py", "requirements.txt", "README.md", "model_specs.json",\n)\n\n\ndef _failure_report(error, stage):\n    """Report diagnostic categories without copying raw text or exception values."""\n    category = "unclassified"\n    reason = "실행 중 오류가 발생했습니다. 실패 단계와 코드 위치를 포함한 오류 보고서를 확인하세요."\n    try:\n        message = str(error)\n    except Exception:\n        message = ""\n    if stage == "settings_load":\n        category = "settings_unreadable"\n        reason = "실행 설정을 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif isinstance(error, ImportError):\n        category = "dependency_import_failed"\n        reason = "필요한 패키지를 불러오지 못했습니다. 패키지 준비 셀의 설치 결과를 확인하세요."\n    elif ("Colab에서 GPU 런타임을 선택하세요" in message\n          or "CUDA GPU" in message or "GPU required" in message):\n        category = "gpu_unavailable"\n        reason = "GPU가 연결되지 않았습니다. Colab 런타임 유형을 GPU로 설정한 뒤 처음부터 실행하세요."\n    elif "CUDA out of memory" in message or type(error).__name__ == "OutOfMemoryError":\n        category = "out_of_memory"\n        reason = "GPU 메모리가 부족합니다. 배치 측정 기록과 GPU 메모리를 확인하고 다른 실행을 종료하세요. 가장 작은 배치도 실패하면 GPU 또는 입력 길이 검토가 필요합니다. 재실행에는 추가 리소스가 듭니다."\n    elif "Input ZIP is missing:" in message:\n        category = "input_zip_missing"\n        reason = "입력 ZIP을 찾지 못했습니다. Drive 연결과 SETTINGS[\'input_zip\']의 실제 경로를 확인하세요."\n    elif isinstance(error, ValueError) and any(token in message.lower() for token in (\n        "source_run", "completed pilot", "pilot completion marker", "pilot source",\n        "pilot recipe", "pilot split", "pilot subset", "pilot selection", "invalid selected pilot epoch",\n    )):\n        category = "pilot_source_invalid"\n        reason = "전체 학습의 출처를 확인하지 못했습니다. source_run을 pilot_complete.json이 있는 완료된 예비 실행 폴더로 지정하세요. 예비 결과의 파일을 빠짐없이 보관하고 학습 데이터·분할·학습 조건을 변경하지 마세요."\n    elif type(error).__name__ == "BudgetExceeded":\n        category = "time_budget_exhausted"\n        reason = "설정한 작업 예산으로 현재 실행 단계를 완료하기 어렵거나 시간이 끝나 중단했습니다. 저장된 시간 추정을 확인하세요. 완료 표시가 없는 결과는 제출에 사용하지 마세요."\n    elif any(token in message for token in (\n        "huggingface.co", "Connection error", "Can\'t load tokenizer", "Can\'t load the model",\n        "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",\n    )) or type(error).__name__ in {"GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError"}:\n        category = "pretrained_model_unavailable"\n        reason = "고정된 공개 모델 또는 토크나이저를 내려받지 못했습니다. Colab 인터넷 연결과 model_specs.json의 모델 공개 상태를 확인하세요."\n    elif stage == "source_read":\n        category = "runtime_sources_unreadable"\n        reason = "노트북에 포함된 실행 코드 일부를 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif stage == "result_write":\n        category = "result_record_failed"\n        reason = "실행 결과 위치를 기록하지 못했습니다. Drive 결과 폴더의 완료 표시와 오류 보고서를 확인하세요."\n    frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,\n               "function": frame.f_code.co_name}\n              for frame, line in traceback.walk_tb(error.__traceback__)]\n    report = {"status": "failed", "stage": stage, "error_type": type(error).__name__,\n              "category": category, "reason": reason, "frames": frames[-20:]}\n    if (isinstance(error, ImportError) and isinstance(error.name, str)\n            and re.fullmatch(r"[A-Za-z_]\\w*(?:\\.[A-Za-z_]\\w*)*", error.name, flags=re.ASCII)):\n        report["module_name"] = error.name\n    if isinstance(error, OSError) and type(error.errno) is int:\n        report["errno"] = error.errno\n    return report\n\n\ndef _write_report(path, report):\n    path.write_text(json.dumps(report, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    error_file = root / "notebook_error.json"\n    stage = "initialize"\n    try:\n        _write_report(error_file, {"status": "starting", "stage": stage})\n        stage = "settings_load"\n        settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n        if not isinstance(settings, dict):\n            raise ValueError("Notebook settings must be a JSON object")\n        stage = "runtime_import"\n        name = "mission3_ver25_execution"\n        spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                      submodule_search_locations=[str(root)])\n        package = importlib.util.module_from_spec(spec)\n        sys.modules[name] = package\n        spec.loader.exec_module(package)\n        runtime = importlib.import_module(name + ".runtime")\n        stage = "source_read"\n        sources = {name: (root / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n        stage = "runtime_run"\n        result = runtime.run(settings, runtime_sources=sources)\n        stage = "result_write"\n        Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n        _write_report(error_file, {"status": "complete", "stage": "complete"})\n    except Exception as error:\n        report = _failure_report(error, stage)\n        try:\n            _write_report(error_file, report)\n        except OSError:\n            report["diagnostic_file_saved"] = False\n        print(json.dumps(report, ensure_ascii=False, allow_nan=False), flush=True)\n        raise SystemExit(1) from None\n\n\nif __name__ == "__main__":\n    main()\n', 'build_notebook.py': '"""Build the standalone Colab Qwen QLoRA comparison notebook."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport textwrap\n\nHERE = Path(__file__).resolve().parent\nSOURCE_FILES = (\n    "__init__.py", "data.py", "analysis.py", "bootstrap.py", "budget.py", "batching.py", "trainer.py", "runtime.py",\n    "inference.py", "run_notebook.py", "build_notebook.py", "requirements.txt", "README.md", "model_specs.json",\n)\n\n\ndef default_settings():\n    return {\'version\': \'ver25\',\n     \'run_mode\': \'pilot\',\n     \'pilot_samples\': 4000,\n     \'source_run\': \'\',\n     \'input_zip\': \'/content/drive/MyDrive/mission3_ver5_inputs.zip\',\n     \'output_dir\': \'/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver25\',\n     \'work_dir\': \'/content/mission3_ver25\',\n     \'train_dir\': \'\',\n     \'validation_dir\': \'\',\n     \'run_validation\': True,\n     \'seed\': 42,\n     \'fit_fraction\': 0.7,\n     \'development_fraction\': 0.15,\n     \'confirmation_fraction\': 0.15,\n     \'qwen_batch_size\': 1,\n     \'qwen_gradient_accumulation\': 16,\n     \'qwen_max_length\': 1024,\n     \'control_batch_size\': 8,\n     \'control_gradient_accumulation\': 2,\n     \'control_max_length\': 512,\n     \'max_epochs\': 1,\n     \'benchmark_examples\': 64,\n     \'benchmark_warmup_examples\': 16,\n     \'effective_batch_size\': 16,\n     \'warmup_ratio\': 0.05,\n     \'lora_lr\': 0.0001,\n     \'head_lr\': 0.0003,\n     \'control_lr\': 2e-05,\n     \'weight_decay\': 0.01,\n     \'lora_r\': 16,\n     \'lora_alpha\': 32,\n     \'lora_dropout\': 0.05,\n     \'gradient_clip\': 1.0,\n     \'min_macro_gain\': 0.01,\n     \'allow_cpu\': False,\n     \'time_budget_minutes\': 180,\n     \'finalization_reserve_seconds\': 1500,\n     \'budget_safety_factor\': 1.35,\n     \'log_every\': 100}\n\n\ndef make_notebook():\n    sources = {name: (HERE / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n    for name, source in sources.items():\n        if name.endswith(".py"):\n            compile(source, name, "exec")\n    cells = []\n\n    def add(kind, identifier, source, form=False):\n        cell = {"cell_type": kind, "id": f"ver25-{identifier}", "metadata": {},\n                "source": (textwrap.dedent(source).strip() + "\\n").splitlines(True)}\n        if kind == "code":\n            cell.update(execution_count=None, outputs=[])\n            if form:\n                cell["metadata"]["cellView"] = "form"\n        cells.append(cell)\n\n    add(\'markdown\', \'intro\', """\n# Mission 3 ver25 — 4,000개 예비 학습 후 결과 저장·종료\n\nColab GPU 런타임에서 **런타임 → 모두 실행**을 누르세요.\nDrive 연결 시 입력 ZIP이 있는 계정을 선택하세요. 별도 코드 업로드 없이 이 노트북과\n기존 `mission3_ver5_inputs.zip`으로 실행합니다. 공개 사전학습 가중치는 실행 중 내려받습니다.\n준비 셀이 실패하거나 런타임을 초기화했다면 마지막 셀만 실행하지 말고 처음부터 실행하세요.\n\n**Qwen3-4B-Instruct-2507의 내부 LoRA adapter와 9개 증상 분류층**을 실제 Training 정답으로 학습합니다.\n새로 학습한 KLUE 대조군, Qwen 단독, 고정 1:1 확률 평균을 같은 분할에서 비교합니다.\n자유 문장 생성이나 외부 LLM 정답 생성은 하지 않습니다. 외부 학습 데이터도 추가하지 않습니다.\n기본 실행은 Training fit 분할에서 중복 그룹과 증상 비율을 고려한 **약 4,000개로 두 모델을 1 epoch 학습**합니다.\n중복 그룹을 나눌 수 없는 경우 실제 개수가 조금 달라질 수 있으며 선택 내역을 저장합니다.\n고정 development에서 비교한 뒤 결과를 저장하고 종료합니다. **전체 학습은 자동으로 이어서 실행하지 않습니다.**\n원본 Training 표본과 정답을 유지합니다. 9개 밖의 라벨만 제외하며 해당 통화는 남깁니다.\n모델 특징은 배열 순서의 `utterances[].text`뿐이며 줄바꿈으로 발화 경계를 유지합니다.\n\n**기본 180분 작업 예산에는 설치·다운로드·준비·속도 측정·학습·평가·저장이 모두 포함됩니다.**\nDrive 로그인 대기 시간은 제외합니다. `time_budget_minutes`로 실행 전에 직접 늘릴 수 있고 자동 연장하지 않습니다.\n**시간 제한 없이 실행하려면 `SETTINGS["time_budget_minutes"] = None`으로 설정하세요.**\n이때 예상 시간에 따른 시작 보류, 설치·실행 시간 제한, 시간 만료 GPU 반환 타이머를 끕니다.\n완료·오류 시 결과 기록과 GPU 반환은 유지하며, Colab 자체의 세션 종료는 막을 수 없습니다.\n동일한 대표 표본으로 배치 후보들의 속도와 메모리를 측정한 뒤 현재 GPU에서 가장 빠른 성공 후보를 선택합니다.\n시간 제한을 켠 경우 마지막 25분은 저장·재로딩·오류 보고·패키지 검증에 예약합니다.\n예산이 부족하면 학습 시작을 보류하거나 미완료 상태로 저장합니다. 각 실행은 추가 GPU 리소스를 소비합니다.\n**변경된 자동 배치 선택과 예비 학습의 GPU 실행 시간·F1 개선은 아직 확인되지 않았습니다.**\n    """)\n\n    add(\'markdown\', \'experiments\', """\n## 고정 실험 조건\n\n| 항목 | Qwen 후보 | 새 KLUE 대조군 |\n|---|---|---|\n| 시작 가중치 | Qwen3-4B-Instruct-2507 | klue/roberta-base |\n| 학습 | NF4 4-bit, double quantization, LoRA + 분류층 | 전체 encoder + 분류층 |\n| 입력 길이 | 1,024, 앞/뒤 보존 + 공통 판정 접미문 | 특수 토큰 포함 앞부분 512 |\n| 자동 측정 batch 후보 | 1 / 2 / 4 / 8 | 4 / 8 / 16 |\n| 유효 batch | 16 (누적 횟수 자동 조정) | 16 (누적 횟수 자동 조정) |\n| 학습률 | LoRA 1e-4 / head 3e-4 | 2e-5 |\n\n9개 독립 logits에서 weighted BCE를 직접 계산합니다.\n`pos_weight = clip(sqrt(음성 수 / max(양성 수, 1)), 1, 5)`는 실제 예비 학습 subset만으로 계산하고,\n최종 refit에서는 전체 Training으로 다시 계산합니다. 판정은 항상 `logit >= 0`입니다.\nLoRA r=16, alpha=32, dropout=0.05, attention/MLP projection 전체를 학습합니다.\n양자화 원본은 고정합니다. bf16 미지원 GPU의 fp16 경로는 pilot에서 먼저 시험합니다.\n\n정규화한 텍스트 중복 그룹을 보존하여 Training을 새 **70% fit / 15% development / 15% confirmation**으로 나눕니다.\n라벨 비율은 가능한 한 유지하며 실제 분할과 데이터 digest를 저장합니다.\n과거 전체 Training 모델은 이 비교에 사용하지 않습니다. 새로운 외부 검증셋이라는 뜻도 아닙니다.\n기본 예비 실행은 1 epoch이며 같은 subset으로 학습한 두 모델을 고정 development에서 비교합니다.\n소규모 1 epoch 결과만으로 전체 데이터 성능이나 수렴 여부를 단정하지 않습니다.\nQwen 또는 사전에 정한 1:1 확률 평균이 대조군보다 0.01 이상 높으면 추진 후보로 삼습니다.\n0.01은 자원 배분 기준이며 통계적 유의성이나 목표 점수 보장이 아닙니다.\n**Macro F1이 우선이며 오심 소폭 하락만으로 후보를 자동 탈락시키지 않습니다.**\n\n**예비 실행에서는 confirmation을 평가하지 않고 공식 Validation을 모델 평가에 사용하지 않습니다.**\n결과를 검토한 뒤 별도 full 실행을 시작하면 예비 실행에서 동결한 선택을 confirmation에서 한 번 비교합니다.\n이 점수로 후보를 다시 고르지 않으며 선택 recipe를 공개 초기 가중치에서 전체 Training으로 새로 학습합니다.\n9개 F1/AP, 오심 P/R/TP/FP/FN, 공통 오답·새 FP/FN, 통화 단위 bootstrap을 저장합니다.\n공식 Validation은 후보 선택에 사용하지 않고 전체 Training refit 이후 한 번 확인합니다.\n이미 반복 관찰된 공식 Validation 점수를 완전히 독립된 일반화 평가라고 부르지 않습니다.\nthreshold/offset/calibration/혼합비 검색, 재라벨링, 삭제, 정답 마스킹은 하지 않습니다.\n첫 실행은 **weighted BCE만** 사용합니다. ASL 재학습과 3-fold OOF는 후속 실험이며 자동 실행하지 않습니다.\n친구 11개 모델 앙상블은 보존할 별도 baseline이며 이번 새 KLUE 대조군과 다릅니다.\n    """)\n\n    add(\'code\', \'settings\', """\n# @title 실행 설정\nSETTINGS = __SETTINGS__\n# First run: keep run_mode=\'pilot\'. Later: set run_mode=\'full\', source_run to\n# the completed pilot run folder. Set time_budget_minutes=None for no time limit,\n# or use a positive integer from its time estimate for a bounded run.\nAUTO_RELEASE_GPU = True # @param {type:"boolean"}\nRUNTIME_PREPARED = None\nDEPENDENCIES_PREPARED = None\n# Each rerun starts a new budget and consumes additional Colab resources.\n    """.replace("__SETTINGS__", repr(default_settings())))\n\n    add(\'code\', \'drive\', """\nfrom google.colab import drive\ndrive.mount("/content/drive", force_remount=False)\n    """)\n\n    add(\'code\', \'budget\', """\n# @title 설정한 작업 예산 시작 (이후 패키지 설치 시간도 포함)\nimport time\nimport threading\nimport math\nRUNTIME_PREPARED = None\nDEPENDENCIES_PREPARED = None\nif not isinstance(globals().get("SETTINGS"), dict) or "AUTO_RELEASE_GPU" not in globals():\n    raise RuntimeError("실행 설정이 없습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")\nconfigured_minutes = SETTINGS["time_budget_minutes"]\nif configured_minutes is not None and (\n        isinstance(configured_minutes, bool) or not isinstance(configured_minutes, (int, float))\n        or not math.isfinite(configured_minutes) or configured_minutes <= 0):\n    raise ValueError("time_budget_minutes must be a finite positive number or None")\nBUDGET_STARTED_MONOTONIC = time.monotonic()\nDEADLINE_MONOTONIC = (None if configured_minutes is None else\n                      BUDGET_STARTED_MONOTONIC + float(configured_minutes) * 60)\nSETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC\nSETTINGS["invocation_started_monotonic"] = BUDGET_STARTED_MONOTONIC\nprevious_timer = globals().get("RELEASE_TIMER")\nif previous_timer is not None:\n    previous_timer.cancel()\nRELEASE_TIMER = None\nRELEASE_LOCK = threading.Lock()\nRELEASE_REQUESTED = False\n\ndef save_setup_failure(error, stage):\n    # Never record exception text: it can contain input text or private paths.\n    import json\n    import traceback\n    import uuid\n    from datetime import datetime, timezone\n    from pathlib import Path\n    frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,\n               "function": frame.f_code.co_name}\n              for frame, line in traceback.walk_tb(error.__traceback__)]\n    report = {"status": "failed", "stage": stage, "error_type": type(error).__name__,\n              "category": "preparation_failed", "frames": frames[-20:]}\n    print("준비 실패 단계:", stage, "· 오류 종류:", type(error).__name__, flush=True)\n    try:\n        folder = Path(SETTINGS["output_dir"]) / "_diagnostics"\n        folder.mkdir(parents=True, exist_ok=True)\n        path = folder / ("ver25_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n                         + "_" + uuid.uuid4().hex[:8] + ".json")\n        path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n        print("오류 보고서:", path, flush=True)\n    except OSError as save_error:\n        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, flush=True)\n\ndef release_gpu():\n    global RELEASE_REQUESTED\n    if not AUTO_RELEASE_GPU:\n        return\n    with RELEASE_LOCK:\n        if RELEASE_REQUESTED:\n            return\n        RELEASE_REQUESTED = True\n        if RELEASE_TIMER is not None:\n            RELEASE_TIMER.cancel()\n    try:\n        from google.colab import runtime as colab_runtime\n        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)\n        colab_runtime.unassign()\n    except Exception as error:\n        print("자동 반환 요청 실패:", type(error).__name__,\n              "· Colab 메뉴 \'런타임 → 연결 해제 및 런타임 삭제\'를 직접 실행하세요.", flush=True)\n\nif AUTO_RELEASE_GPU and DEADLINE_MONOTONIC is not None:\n    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)\n    RELEASE_TIMER.daemon = True\n    RELEASE_TIMER.start()\nif DEADLINE_MONOTONIC is None:\n    print("작업 시간 제한: 없음 · 완료 또는 오류 시 결과 기록 후 GPU 반환")\nelse:\n    print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")\n    """)\n\n    add(\'code\', \'restore\', """\n# @title 실행 코드 준비 (코드 파일 추가 업로드 불필요)\nimport importlib.util\nimport sys\nimport tempfile\nfrom pathlib import Path\nRUNTIME_SOURCES = __SOURCES__\nRUNTIME_PREPARED = None\nDEPENDENCIES_PREPARED = None\nif ("DEADLINE_MONOTONIC" not in globals() or "time" not in globals()\n        or not callable(globals().get("release_gpu"))):\n    raise RuntimeError("작업 예산 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")\ntry:\n    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:\n        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")\n    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver25_"))\n    for name, source in RUNTIME_SOURCES.items():\n        (runtime_dir / name).write_bytes(source.encode("utf-8"))\n    budget_spec = importlib.util.spec_from_file_location("mission3_ver25_budget_supervisor", runtime_dir / "budget.py")\n    budget_module = importlib.util.module_from_spec(budget_spec)\n    sys.modules[budget_spec.name] = budget_module\n    budget_spec.loader.exec_module(budget_module)\n    RUNTIME_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)\n    print("실행 코드 준비 완료")\nexcept BaseException as setup_error:\n    reporter = globals().get("save_setup_failure")\n    if callable(reporter):\n        reporter(setup_error, "source_restore")\n    cleanup = globals().get("release_gpu")\n    if callable(cleanup):\n        try:\n            cleanup()\n        except Exception as cleanup_error:\n            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")\n    raise RuntimeError("실행 코드 준비 실패: " + type(setup_error).__name__ + ". Drive 오류 보고서를 확인하세요.") from None\n    """.replace("__SOURCES__", repr(sources)), form=True)\n\n    add(\'code\', \'dependencies\', """\n# @title ver25 전용 QLoRA 패키지 준비 (시간 제한 사용 시 최대 20분, 전체 예산에 포함)\nDEPENDENCIES_PREPARED = None\nrequired = ("runtime_dir", "budget_module", "DEADLINE_MONOTONIC", "time", "sys")\nif (any(name not in globals() for name in required)\n        or not callable(globals().get("release_gpu"))):\n    raise RuntimeError("실행 코드 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")\nif (globals().get("RUNTIME_PREPARED") != (str(runtime_dir), DEADLINE_MONOTONIC)\n        or not (runtime_dir / "requirements.txt").is_file()):\n    raise RuntimeError("현재 실행의 코드 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")\ntry:\n    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:\n        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")\n    package_deadline = (None if DEADLINE_MONOTONIC is None else\n                        min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60))\n    budget_module.run_supervised(\n        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],\n        deadline_monotonic=package_deadline, shutdown_margin_s=10,\n    )\n    DEPENDENCIES_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)\n    if DEADLINE_MONOTONIC is None:\n        print("패키지 준비 완료 · 작업 시간 제한 없음")\n    else:\n        print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")\nexcept BaseException as setup_error:\n    reporter = globals().get("save_setup_failure")\n    if callable(reporter):\n        reporter(setup_error, "dependency_install")\n    cleanup = globals().get("release_gpu")\n    if callable(cleanup):\n        try:\n            cleanup()\n        except Exception as cleanup_error:\n            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")\n    raise RuntimeError("패키지 준비 실패: " + type(setup_error).__name__ + ". Drive 오류 보고서를 확인하세요.") from None\n    """)\n\n    add(\'code\', \'run\', """\n# @title 실행 모드에 따라 학습 → 결과 저장 → 종료 · GPU 반환\nimport csv\nimport json\nimport math\nimport subprocess\nrequired = ("runtime_dir", "SETTINGS", "budget_module", "DEADLINE_MONOTONIC",\n            "BUDGET_STARTED_MONOTONIC", "time", "Path", "sys")\nif (any(name not in globals() for name in required)\n        or not callable(globals().get("release_gpu"))):\n    raise RuntimeError("준비 셀이 완료되지 않았거나 런타임이 초기화됐습니다. 마지막 셀만 실행하지 말고 런타임 → 모두 실행으로 처음부터 실행하세요. 앞선 셀이 실패했다면 그 셀의 첫 오류를 확인하세요.")\nprepared_context = (str(runtime_dir), DEADLINE_MONOTONIC)\nif (globals().get("RUNTIME_PREPARED") != prepared_context\n        or globals().get("DEPENDENCIES_PREPARED") != prepared_context\n        or not (runtime_dir / "run_notebook.py").is_file()):\n    raise RuntimeError("현재 실행의 코드 또는 패키지 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")\nerror_file = runtime_dir / "notebook_error.json"\ntry:\n    error_file.write_text(json.dumps({"status": "starting"}), encoding="utf-8")\n    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:\n        raise budget_module.BudgetExceeded("notebook launch", deadline_monotonic=DEADLINE_MONOTONIC)\n    settings_file = runtime_dir / "notebook_settings.json"\n    result_file = runtime_dir / "notebook_result.json"\n    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    budget_module.run_supervised(\n        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),\n         "--settings", str(settings_file), "--result_file", str(result_file)],\n        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,\n    )\n    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))\n    print("결과 폴더:", result_dir)\n    summary = result_dir / "candidate_summary.csv"\n    if summary.is_file():\n        print("동일 Training development 분할 비교:")\n        with summary.open(encoding="utf-8-sig", newline="") as handle:\n            for row in csv.DictReader(handle):\n                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),\n                      "오심 F1 =", round(float(row["nausea_f1"]), 6),\n                      "선택 epoch =", row.get("selected_epochs", "기록 확인"))\n    for filename in ["training_metrics.json", "validation_metrics.json"]:\n        path = result_dir / filename\n        if path.is_file():\n            report = json.loads(path.read_text(encoding="utf-8-sig"))\n            print(filename, "Macro F1 =", round(report["macro_f1"], 6))\n            for symptom, values in report["per_symptom"].items():\n                print(symptom, "F1 =", round(values["f1"], 4),\n                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))\n    manifest_path = result_dir / "mission3_ver25_model.ckpt"\n    manifest = {}\n    if manifest_path.is_file():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))\n        selection = manifest.get("selection", {})\n        print("저장된 후보:", selection.get("selected_candidate", "모델 정보 확인"))\n        print("저장 모델 학습 범위:", manifest.get("training_scope", "모델 정보 확인"),\n              "· 학습 표본:", manifest.get("fitted_samples", "모델 정보 확인"),\n              "· 학습 epoch:", selection.get("selected_epochs", "모델 정보 확인"))\n    status_path = result_dir / "runtime.json"\n    status = json.loads(status_path.read_text(encoding="utf-8-sig")) if status_path.is_file() else {}\n    print("실행 상태:", status.get("status", "기록 확인 필요"))\n    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")\n    print("선택 근거:", result_dir / "final_selection.json")\n    components = manifest.get("components", [])\n    components_present = bool(components) and all(\n        isinstance(item, dict) and isinstance(item.get("model_directory"), str)\n        and (result_dir / item["model_directory"]).is_dir() for item in components)\n    complete = ((result_dir / "run_complete.json").is_file() and manifest_path.is_file()\n                and manifest.get("training_scope") == "all_training" and components_present\n                and ((result_dir / "mission3.csv").is_file() or not SETTINGS["run_validation"])\n                and status.get("status") in {"complete", "complete_not_improved"})\n    pilot_path = result_dir / "pilot_complete.json"\n    pilot = json.loads(pilot_path.read_text(encoding="utf-8-sig")) if pilot_path.is_file() else {}\n    pilot_complete = (SETTINGS["run_mode"] == "pilot" and status.get("status") == "pilot_complete"\n                      and pilot.get("status") == "pilot_complete"\n                      and pilot.get("training_scope") == "training_subset"\n                      and pilot.get("submission_ready") is False\n                      and all((result_dir / name).is_file() for name in\n                              ("candidate_summary.csv", "final_selection.json", "full_run_estimate.json"))\n                      and not manifest_path.exists() and not (result_dir / "run_complete.json").exists())\n    if pilot_complete:\n        fitted_samples = pilot.get("fitted_samples", SETTINGS["pilot_samples"])\n        print("예비 학습 완료 ·", fitted_samples, "개 학습 결과를 저장하고 종료했습니다.")\n        print("전체 Training 학습과 confirmation·공식 Validation 평가는 실행하지 않았습니다.")\n        print("후속 전체 학습 예상 시간:", result_dir / "full_run_estimate.json")\n        estimate = json.loads((result_dir / "full_run_estimate.json").read_text(encoding="utf-8-sig"))\n        recommended_minutes = estimate.get("recommended_budget_minutes") if isinstance(estimate, dict) else None\n        if (isinstance(recommended_minutes, (int, float)) and not isinstance(recommended_minutes, bool)\n                and math.isfinite(recommended_minutes) and recommended_minutes > 0):\n            print("예상 권장 전체 실행 예산:", recommended_minutes, "분 (약",\n                  round(recommended_minutes / 60, 1), "시간) · 실측 기반 추정이며 보장 시간은 아닙니다.")\n        print("결과를 검토한 뒤 SETTINGS의 run_mode=\'full\', source_run=위 결과 폴더,")\n        print("time_budget_minutes=None(시간 제한 없음) 또는 예상 시간을 반영한 예산으로 설정하고 처음부터 모두 실행하세요.")\n        print("예비 학습 결과는 최종 제출 모델이 아닙니다.")\n    elif complete and SETTINGS["run_mode"] == "full":\n        if SETTINGS["run_validation"]:\n            print("최종 작업 완료 · 전체 Training 재학습과 저장 검증을 마친 예측:", result_dir / "mission3.csv")\n        else:\n            print("모델 검증 실행 완료 · Validation 예측과 제출 CSV는 만들지 않았습니다.")\n            print("제출용 실행은 run_validation=True로 설정해야 합니다.")\n        print("이전 최고 모델과의 성능 비교는 validation_metrics.json으로 따로 확인하세요.")\n        if status.get("status") == "complete_not_improved":\n            print("새 표현 후보가 대조군보다 개선되지 않음 · 기존 baseline보다 개선한 결과로 표시하지 않습니다.")\n    else:\n        print("미완료 실행:", status.get("status", "상태 기록 없음"),\n              "· 최종 제출 결과로 사용하지 마세요. 결과 폴더의 로그를 확인하세요.")\nexcept Exception as worker_error:\n    failure = {"status": "failed", "stage": "notebook_run", "error_type": type(worker_error).__name__,\n               "category": "notebook_execution_failed",\n               "reason": "실행 또는 결과 확인 중 오류가 발생했습니다. 실패 단계와 코드 위치를 확인하세요.", "frames": []}\n    if isinstance(worker_error, subprocess.CalledProcessError):\n        failure.update(stage="worker_process", category="worker_process_failed", exit_code=worker_error.returncode,\n                       reason="실행 프로그램이 상세 오류를 저장하기 전에 종료됐습니다. 종료 코드와 준비 상태 확인이 필요합니다.")\n        try:\n            recorded = json.loads(error_file.read_text(encoding="utf-8"))\n            if isinstance(recorded, dict) and recorded.get("status") == "failed":\n                failure.update(recorded)\n        except (OSError, ValueError):\n            pass\n    elif type(worker_error).__name__ == "BudgetExceeded":\n        failure.update(category="time_budget_exhausted",\n                       reason="설정한 작업 예산이 끝나 실행을 중단했습니다. 완료 표시가 없는 결과는 제출에 사용하지 마세요.")\n    print("ver25 내부 오류 · 실패 단계:", failure["stage"], "· 오류 종류:", failure["error_type"], flush=True)\n    print("확인 내용:", failure["reason"], flush=True)\n    if failure.get("module_name"):\n        print("불러오지 못한 모듈:", failure["module_name"], flush=True)\n    for frame in failure.get("frames", [])[-6:]:\n        print("코드 위치:", frame["file"], "·", frame["function"], "·", frame["line"], flush=True)\n    saved_error = None\n    try:\n        from datetime import datetime, timezone\n        import uuid\n        error_dir = Path(SETTINGS["output_dir"]) / "_diagnostics"\n        error_dir.mkdir(parents=True, exist_ok=True)\n        saved_error = error_dir / ("ver25_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n                                  + "_" + uuid.uuid4().hex[:8] + ".json")\n        saved_error.write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8")\n        print("오류 보고서:", saved_error, flush=True)\n    except OSError as save_error:\n        saved_error = None\n        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, "· 위의 내부 오류 내용을 복사하세요.", flush=True)\n    raise RuntimeError("ver25 실행 실패 [" + failure["stage"] + " / " + failure["error_type"] + "]: "\n                       + failure["reason"] + (" 오류 보고서: " + str(saved_error) if saved_error else "")) from None\nfinally:\n    cleanup = globals().get("release_gpu")\n    if callable(cleanup):\n        try:\n            cleanup()\n        except Exception as cleanup_error:\n            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· Colab에서 직접 연결을 해제하세요.")\n    """)\n\n    add(\'markdown\', \'results\', """\n## 결과와 완료 상태\n\n결과는 `DCC_AIdeas/results/mission3/ver25/실행시각_고유번호/`에 저장합니다.\n- `model_specs.json`, split/data digests: 정확한 모델 revision과 학습 데이터 출처\n- `candidate_summary.csv`, `final_selection.json`: development 선택과 고정 recipe\n- development 예측 및 지표: 9개 F1/AP, 오심 지표, 공통 오답 변화와 불확실성\n- 배치 속도 측정 기록: 후보별 처리량·GPU 메모리·선택 결과\n- `runtime.json`: 실행 상태, 실측 시간, 예산 판단\n- `pilot_complete.json`: 예비 학습 저장 검증 완료, `submission_ready=false`\n- `full_run_estimate.json`: 이후 전체 학습에 필요한 시간의 실측 기반 추정\n\n**기본 pilot는 결과 저장 후 종료합니다. confirmation·공식 Validation·전체 Training refit은 실행하지 않습니다.**\n작은 subset의 development 점수는 확장 여부를 판단하는 자료이며 전체 학습의 최종 성능이 아닙니다.\n예비 체크포인트와 고정 모델 revision을 포함하므로 실행 폴더 전체를 보관하세요.\n\n## 결과를 보고 전체 학습을 따로 실행하기\n\n1. 예비 결과의 F1·오심 지표와 `full_run_estimate.json`을 확인합니다.\n2. 위의 **실행 설정 셀 하나에서** `run_mode`를 `\'full\'`로 바꿉니다.\n3. `source_run`에는 `pilot_complete.json`이 있는 완료된 예비 실행 폴더의 절대 경로를 넣습니다.\n4. 시간 제한 없이 끝까지 실행하려면 `time_budget_minutes=None`으로 설정합니다. 시간 제한을 원하면 예상 시간과 준비·저장 여유를 반영해 분 단위 정수로 지정합니다. 기본값은 180분입니다.\n5. GPU 런타임에서 **처음부터 모두 실행**합니다. 원래 예비 결과는 보존하고 별도 결과 폴더를 만듭니다.\n\nfull 실행은 원본 데이터·분할·고정 학습 조건과 완료된 예비 실행의 무결성을 확인합니다.\n같은 모델 revision과 동결한 후보·epoch를 사용하고 현재 GPU의 배치를 다시 측정합니다.\nconfirmation은 동결한 선택을 한 번 확인하는 용도이며 후보를 다시 고르지 않습니다.\n작은 subset에서 이어서 학습하는 대신 선택 recipe만 공개 초기 가중치에서 전체 Training으로 새로 학습합니다.\n공식 Validation은 전체 refit 후에만 `run_validation=True`일 때 평가합니다.\n\nfull 실행에서만 `mission3_ver25_model.ckpt`, 최종 모델 폴더, `submission/`을 만듭니다.\n`validation_metrics.json`, `mission3.csv`는 공식 Validation 평가를 켠 full 실행 결과입니다.\n`run_complete.json`은 전체 Training refit과 최종 저장·오프라인 재로딩 검증을 통과했을 때만 만듭니다.\n\n**fit-only / 미완료 결과는 전체 Training 최종 제출 모델이 아닙니다.**\n`complete_not_improved`는 새 표현 후보가 대조군보다 개선되지 않은 실행의 전체 Training refit과 저장 검증을 마친 상태입니다.\n기존 친구 baseline보다 개선했다는 뜻이 아닙니다. 최종 완료 표시는 전체 refit과 저장 검증이 끝났을 때만 만듭니다.\nmanifest 파일 하나만 보관하지 말고 실행 폴더 전체를 보관하세요.\nNF4 Qwen 및 Qwen 포함 앙상블은 저장된 로컬 가중치로 추론할 때도 CUDA GPU가 필요합니다.\nKLUE 단독 모델은 CPU 추론도 지원합니다.\n실패 시 원문이나 개별 데이터 파일명을 출력하지 않고 실패 단계·오류 종류·코드 위치를\n`ver25/_diagnostics/`에 저장한 뒤 GPU 반환을 요청합니다.\n중단된 학습의 자동 재개는 지원하지 않습니다. 자동 반환 실패 시 Colab의 연결 해제/런타임 삭제를 직접 실행하세요.\n\nQwen은 Apache-2.0, KLUE는 CC-BY-SA-4.0이며 출처와 조건을 패키지에 보존합니다.\n의존성은 ver25 전용 `requirements.txt`를 사용합니다. 노트북 의존성 설치는 Colab에서 실행합니다.\n    """)\n    return {\n        "cells": cells,\n        "metadata": {"colab": {"name": "mission3_ver25.ipynb"}, "accelerator": "GPU",\n                     "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n                     "language_info": {"name": "python", "version": "3.12"}},\n        "nbformat": 4, "nbformat_minor": 5,\n    }\n\n\ndef build(output=None):\n    target = Path(output) if output is not None else HERE.parent / "mission3_ver25.ipynb"\n    notebook = make_notebook()\n    for cell in notebook["cells"]:\n        if cell["cell_type"] == "code":\n            compile("".join(cell["source"]), cell["id"], "exec")\n    target.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\\n", encoding="utf-8")\n    return target\n\n\nif __name__ == "__main__":\n    print(build())\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\npeft==0.17.1\nbitsandbytes==0.48.1\naccelerate==1.10.1\n', 'README.md': '# Mission 3 ver25\n\n기본 실행은 **Training 중 약 4,000개로 Qwen과 새 KLUE 대조군을 각각 1 epoch 학습하고, development 결과 저장 후 종료**합니다. 전체 학습은 결과를 검토한 뒤 별도 `full` 실행으로 시작합니다. Qwen3-4B-Instruct-2507의 QLoRA adapter와 9출력 분류 head를 원본 정답으로 학습하며 Qwen 단독, KLUE 단독, 고정 1:1 확률 평균을 비교합니다. 변경된 자동 배치 선택과 예비 학습의 실제 GPU 실행 시간·성능 개선은 아직 확인되지 않았습니다.\n\n## 첫 실행: 4,000개 예비 학습\n\n1. `../mission3_ver25.ipynb`를 Colab에 업로드하고 GPU 런타임을 선택합니다.\n2. 기존 `mission3_ver5_inputs.zip`이 있는 계정으로 Drive를 연결합니다. 다른 위치라면 설정의 `input_zip`만 수정합니다.\n3. 기본 설정 `run_mode=\'pilot\'`, `pilot_samples=4000`, `max_epochs=1`을 유지하고 **런타임 → 모두 실행**을 누릅니다. 코드가 노트북에 포함되어 있어 저장소 전체 업로드는 필요 없습니다.\n4. 결과는 `MyDrive/DCC_AIdeas/results/mission3/ver25/실행시각_고유번호/`에 저장합니다.\n\nDrive 연결 직후, 패키지 설치 전에 작업 예산을 시작합니다. 기본값은 180분이며 `time_budget_minutes`를 실행 전에 유한한 양수로 직접 지정해 늘릴 수 있습니다. 자동 연장하지 않습니다. 기본 25분은 저장·재로딩·결과 검증용으로 예약합니다. 설치·다운로드·준비·속도 측정·학습·평가도 예산에 포함됩니다. 학습은 설치 후 새 Python 프로세스에서 시작합니다. 마지막 셀만 실행하거나 실패한 준비 셀을 건너뛸 수 없습니다. 재실행은 새 실행이며 추가 GPU 리소스를 소비합니다.\n\n**시간 제한 없이 실행하려면 기존 설정 셀 마지막에 `SETTINGS["time_budget_minutes"] = None`을 추가합니다.** 문자열 `"None"`이나 `0`은 사용할 수 없습니다. 이 설정은 예상 시간에 따른 학습 시작 보류, 실행 중 deadline 검사, 패키지 설치와 학습 프로세스의 시간 제한, 시간 만료 GPU 반환 타이머를 끕니다. 배치 속도 측정과 예상 시간 기록은 계속 수행합니다. `AUTO_RELEASE_GPU=True`는 유지하면 됩니다. 완료·오류 시 결과 기록 후 GPU를 반환하며, Colab 자체의 세션 종료는 막을 수 없습니다. 학습 epoch 수와 pilot/full 구분은 그대로 적용됩니다. 결과 JSON의 `remaining_seconds: null`은 시간 제한이 없다는 뜻입니다.\n\n고정된 fit 20,440개 중 중복 그룹과 증상 비율을 고려하여 약 4,000개를 뽑고 두 모델에 같은 표본을 사용합니다. 중복 그룹을 나누지 않으므로 표본 수가 목표에서 조금 벗어날 수 있으며 실제 수와 선택 ID를 저장합니다. development 4,380개는 그대로 유지합니다. **pilot에서는 confirmation을 평가하지 않고, 공식 Validation을 모델 평가에 사용하지 않으며 전체 Training refit을 시작하지 않습니다.** 입력 ZIP 준비 시 Validation 파일도 추출·무결성 검사에 포함될 수 있습니다. `run_validation=True`도 full 단계에만 적용됩니다.\n\n## 배치 자동 선택\n\nQwen batch 1/2/4/8, KLUE batch 4/8/16을 같은 대표 표본으로 측정합니다. 기본 `benchmark_warmup_examples=16` 이후 `benchmark_examples=64`개에서 실제 forward/backward·optimizer 처리량과 GPU 메모리를 기록합니다. 최대 입력 길이의 메모리 검사도 수행하고 GPU 메모리 여유가 10% 미만이거나 메모리가 부족한 후보를 제외합니다. 성공한 후보 중 가장 빠른 배치를 선택하며 기울기 누적 횟수를 조정하여 `effective_batch_size=16`을 유지합니다. Qwen 1×16, KLUE 8×2 설정값은 측정 전 초기값이며 학습에는 자동 선택 결과를 적용합니다. 배치를 4배로 늘려도 속도가 4배가 된다는 보장은 없습니다.\n\n측정용 모델 업데이트를 본 학습의 시작 상태로 사용하지 않습니다. bf16 지원 시 bf16, 미지원 시 fp16 경로를 측정하며 준비·학습·평가·저장 예상 시간에는 기본 안전계수 1.35를 적용합니다. 시간 제한을 켠 경우 예산이 부족하면 시작을 보류하거나 미완료 결과로 저장하고, 실행 중에도 deadline을 검사합니다. 현재 GPU에서의 실제 처리량으로 시간 추정이 갱신되며 모든 GPU에서 기본 180분 내 완료를 보장하지 않습니다.\n\n## 결과를 보고 전체 학습을 따로 실행하기\n\n1. 완료된 예비 실행의 `candidate_summary.csv`, `final_selection.json`, development 지표·예측을 확인합니다. `pilot_complete.json`이 있어야 full 실행의 출처로 사용할 수 있습니다.\n2. 같은 폴더의 `full_run_estimate.json`에서 전체 실행 예상 시간을 확인합니다. GPU·네트워크·저장 속도가 바뀌면 실제 시간도 달라집니다.\n3. 노트북의 **기존 실행 설정 셀 하나에서** `run_mode`를 `\'full\'`로 바꿉니다.\n4. `source_run`을 완료된 예비 실행 폴더의 절대 경로로 지정합니다. 예: `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver25/20261010T000000Z_example` (실제 생성된 폴더로 교체).\n5. 시간 제한 없이 끝까지 실행하려면 `time_budget_minutes=None`으로 설정합니다. 시간 제한을 원하면 예상 시간과 준비·저장 여유를 반영해 분 단위 정수로 지정합니다. GPU 런타임에서 처음부터 **모두 실행**합니다. 원래 예비 결과를 덮어쓰지 않고 새 실행 폴더를 만듭니다.\n\n시간 제한 없는 전체 실행은 **수정된 최신 노트북**을 업로드하고 기존 설정 셀의 마지막에 아래를 추가합니다. `source_run`은 `budget_not_started` 실행 폴더가 아니라 `pilot_complete.json`이 있는 완료된 예비 실행 폴더입니다.\n\n```python\nSETTINGS["run_mode"] = "full"\nSETTINGS["source_run"] = "/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver25/20261010T145455Z_3c1d8643"\nSETTINGS["time_budget_minutes"] = None\n```\n\nfull은 출처 실행의 완료 상태, 데이터·분할·고정 학습 조건과 저장 파일의 무결성을 검증합니다. 예비 실행의 고정 모델 revision, 동결한 후보와 epoch를 그대로 사용하고 현재 GPU에 맞춰 배치를 다시 측정합니다. 동결한 선택을 confirmation에서 한 번 확인한 뒤, 선택 recipe만 공개 초기 가중치에서 전체 Training 29,200개로 새로 학습합니다. 작은 subset의 가중치에서 이어서 학습하거나 더 큰 fit 비교를 다시 실행하지 않습니다. 공식 Validation은 전체 refit 후 `run_validation=True`일 때만 평가합니다.\n\n## 고정 조건\n\n- 입력은 배열 순서의 `utterances[].text`만 사용합니다. `utterences`는 기존 데이터 호환 파서만 지원합니다. 발화 경계는 줄바꿈으로 유지합니다. 메타데이터·파일명·음성은 모델 특징에 넣지 않습니다.\n- 라벨 순서는 고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란입니다. 다른 라벨만 제외하며 해당 통화는 유지합니다. 샘플 삭제·재라벨링·정답 마스킹·외부 데이터·생성 정답을 사용하지 않습니다.\n- 텍스트 중복 그룹을 분리하지 않는 새 70/15/15 fit/development/confirmation 분할을 저장합니다. 그룹 제약에 따른 실제 개수와 digest를 기록합니다. 과거 all-Training 가중치는 비교에 사용하지 않습니다.\n- Qwen: NF4 4-bit + double quantization, LoRA r16/alpha32/dropout0.05, 모든 block의 attention/MLP projection과 분류 head 학습, gradient checkpointing, `use_cache=False`, 동적 padding입니다.\n- Qwen 입력은 최대 1,024 tokens이며 앞·뒤 전사와 모든 표본에 같은 판정 접미문을 보존합니다. 마지막 non-padding 판정 위치를 pooling합니다. 길이 분포·초과율을 저장합니다. KLUE는 특수 토큰을 포함한 앞부분 최대 512 tokens를 유지하고 초과한 뒷부분을 자릅니다.\n- 실제 배치는 위의 자동 측정 결과를 사용하며 유효 batch16을 유지합니다. LoRA/head/KLUE learning rate는 각각 1e-4/3e-4/2e-5, warmup0.05, AdamW weight decay0.01, gradient clip1.0, 기본1 epoch입니다.\n- 9개 float32 logits에서 명시적 weighted BCE를 계산합니다. `pos_weight = clip(sqrt(N_negative / max(N_positive, 1)), 1, 5)`는 실제 예비 학습 subset에서만 계산하고 final refit 때 전체 Training으로 다시 계산합니다.\n- 판정은 모든 라벨에 `logits >= 0`입니다. threshold·offset·calibration·혼합비 탐색을 하지 않습니다.\n\n## 선택과 해석\n\n각 모델의 epoch와 recipe는 development Macro F1로 선택합니다. 기본값은 1 epoch이며 소규모 1 epoch의 낮은 점수만으로 수렴 실패를 단정하거나 전체 학습 성능을 보장하지 않습니다. 같은 4,000개로 학습한 새 KLUE 대조군과 비교합니다. 오심 소폭 하락만으로 Macro 개선 후보를 탈락시키지 않습니다. Qwen 또는 고정1:1 평균의 대조군 대비 +0.01은 고비용 확장을 위한 추진 기준이며 통계적 유의성 기준이 아닙니다. 선택을 예비 실행에서 동결한 뒤 별도 full 실행의 confirmation에서 한 번만 비교하고 재선택하지 않습니다.\n\n9개 F1/AP, 오심 P/R/TP/FP/FN, 공통 FP/FN 및 새 오답, 통화 단위 paired bootstrap을 저장합니다. confirmation을 후속 실험 설계에 반복 사용하면 개발셋이 된다는 한계가 있습니다. 이번 분할은 완전히 새로운 외부 검증이 아닙니다.\n\n예산이 남아도 pilot에서 전체 Training 학습을 자동으로 시작하지 않습니다. full의 공식 Validation은 이미 반복 관찰된 셋이라는 한계를 남깁니다. weighted BCE만 구현합니다. ASL과3-fold OOF, 친구11개 앙상블의 새 split 재학습은 별도 예산을 정할 후속 실험입니다.\n\n## 완료 여부와 제출\n\n예비 성공은 `runtime.json`의 `status=\'pilot_complete\'`와 `pilot_complete.json`으로 확인합니다. 학습 체크포인트·development 지표/예측·선택 결과·배치 속도 기록·전체 실행 시간 추정을 저장하며 `training_scope=\'training_subset\'`, `submission_ready=false`입니다. **예비 실행은 정상 완료해도 제출 모델이 아닙니다.** `run_complete.json`과 최종 manifest를 만들지 않습니다.\n\nfull 최종 완료는 `runtime.json`과 `run_complete.json`으로 확인하며 전체 Training refit, 로컬 재로딩, 결과·파일 검증이 모두 성공했을 때만 기록합니다. `complete_not_improved`도 새 표현 후보가 대조군보다 개선되지 않았다는 표시이며 기존 최고 baseline을 이겼다는 뜻이 아닙니다. 작은 양의 개선도 실제 development 최고 후보를 선택하며, +0.01 미만이면 고비용 확장 권고만 끕니다. `fit_only_budget` 등 진단 완료 상태와 `diagnostic_complete.json`은 예비 성공이나 최종 제출 완료가 아닙니다.\n\nfull의 `mission3_ver25_model.ckpt`는 모델 경로와 조합을 담은 JSON manifest입니다. 해당 파일만 옮기지 말고 base·adapter·head·tokenizer·revision·라이선스·실행 코드를 포함한 실행 폴더 전체를 보관합니다. 예비 실행도 출처 검증과 confirmation에 필요한 체크포인트가 있으므로 폴더 전체를 보관해야 합니다. 최종 분류 recipe에 따라 `final_control/`, `final_qwen/`, `shared_base/`가 필요합니다.\n\n제출 폴더의 `inference.py`는 다음 인터페이스를 수용합니다.\n\n```bash\npython inference.py --audio_dir /path/to/wav --label_dir /path/to/json --ckpt_path ../mission3_ver25_model.ckpt --output ./outputs/mission3.csv\n```\n\n`--checkpoint`는 `--ckpt_path`의 별칭입니다. `audio_dir`는 인터페이스 호환용입니다. 추론은 패키지에 포함된 로컬 가중치만 사용하며 다운로드하지 않습니다. **NF4 Qwen 또는 Qwen을 포함한 앙상블은 오프라인 추론에도 CUDA GPU가 필요합니다.** KLUE 단독 모델은 CPU에서도 추론할 수 있습니다. 출력은 `label file name`과 string symptom list입니다.\n\n실패 시 `_diagnostics/ver25_error_*.json`에 실패 단계·오류 종류·코드 위치를 저장하며 예외 메시지에 포함될 수 있는 원문·개인 경로는 출력하지 않습니다. GPU 반환은 완료·오류·시간 제한 시 요청하며 자동 반환 실패 시 Colab에서 직접 연결을 해제하고 런타임을 삭제합니다. 중단된 학습을 자동 재개하지 않습니다.\n\n## 의존성과 라이선스\n\nver25 전용 requirements의 기존 고정 버전을 유지합니다 (`transformers==4.57.1`, `peft==0.17.1`, `bitsandbytes==0.48.1`, `accelerate==1.10.1` 등). 이번 변경으로 의존성을 추가·업그레이드하지 않습니다. 노트북 의존성 설치는 Colab에서 실행하며 로컬 전역 환경은 변경하지 않습니다. 실제 Colab GPU 조합에서 변경된 속도 측정·학습의 검증이 필요합니다.\n\n- [Qwen 공식 모델](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), [Apache-2.0 라이선스](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/blob/main/LICENSE): 실행 시작 시 공개 revision을 한 번 immutable SHA로 고정하여 저장·재사용합니다.\n- [KLUE 공식 모델](https://huggingface.co/klue/roberta-base): 고정 revision `02f94ba5e3fcb7e2a58a390b8639b0fac974a8da`, CC-BY-SA-4.0. 재배포 시 출처·동일조건변경허락 조건을 보존합니다.\n- [PEFT quantization](https://huggingface.co/docs/peft/v0.17.0/en/developer_guides/quantization), [bitsandbytes 0.48.1](https://pypi.org/project/bitsandbytes/0.48.1/), [accelerate 1.10.1](https://pypi.org/project/accelerate/1.10.1/).\n\n## 로컬 검증\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트에서 실행합니다. 학습 의존성이 준비된 검증 환경이 필요합니다.\n\n```bash\npython src/mission3/mission3_ver25/build_notebook.py\npython -m unittest discover -s src/mission3/mission3_ver25 -t . -p "test_*.py" -v\n```\n\n실제 로컬 Training 29,200개를 읽어 새 분할 구현을 확인했으며 fit 20,440개 / development 4,380개 / confirmation 4,380개였습니다. 현재 데이터에서는 예비 subset을 정확히 4,000개로 선택했고 fit 대비 증상별 양성 비율의 최대 차이는 약 0.01101%p였습니다. 이는 데이터 파싱과 분할 검증 결과이며 모델 학습·성능 점수가 아닙니다. 정규화한 텍스트의 동일 중복 그룹을 검사하며 가까운 중복의 유사도 분석은 미확인입니다.\n\n작은 fixture 검증은 원본 데이터·대형 공개 모델 다운로드·GPU 학습을 실행하지 않습니다. 이 테스트 통과만으로 실제 GPU 학습 속도·메모리·정확도·Colab 자동 반환이 확인되지는 않습니다.\n', 'model_specs.json': '{\n  "qwen": {\n    "model_id": "Qwen/Qwen3-4B-Instruct-2507",\n    "revision": "main",\n    "revision_policy": "Resolve once to immutable snapshot commit before training; persist and reuse exact revision.",\n    "model_card": "https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507",\n    "source": "https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507",\n    "license": "Apache-2.0",\n    "license_note": "Apache-2.0 per upstream LICENSE. Bundle base license and notices with redistributed weights.",\n    "license_url": "https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/blob/main/LICENSE"\n  },\n  "control": {\n    "model_id": "klue/roberta-base",\n    "revision": "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da",\n    "source": "https://huggingface.co/klue/roberta-base",\n    "model_card": "https://huggingface.co/klue/roberta-base",\n    "license": "CC-BY-SA-4.0",\n    "license_note": "CC-BY-SA-4.0 per upstream model card; preserve attribution and share-alike terms.",\n    "license_url": "https://huggingface.co/klue/roberta-base"\n  }\n}\n'}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if ("DEADLINE_MONOTONIC" not in globals() or "time" not in globals()
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("작업 예산 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
try:
    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver25_"))
    for name, source in RUNTIME_SOURCES.items():
        (runtime_dir / name).write_bytes(source.encode("utf-8"))
    budget_spec = importlib.util.spec_from_file_location("mission3_ver25_budget_supervisor", runtime_dir / "budget.py")
    budget_module = importlib.util.module_from_spec(budget_spec)
    sys.modules[budget_spec.name] = budget_module
    budget_spec.loader.exec_module(budget_module)
    RUNTIME_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    print("실행 코드 준비 완료")
except BaseException as setup_error:
    reporter = globals().get("save_setup_failure")
    if callable(reporter):
        reporter(setup_error, "source_restore")
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise RuntimeError("실행 코드 준비 실패: " + type(setup_error).__name__ + ". Drive 오류 보고서를 확인하세요.") from None


In [ ]:
# @title ver25 전용 QLoRA 패키지 준비 (시간 제한 사용 시 최대 20분, 전체 예산에 포함)
DEPENDENCIES_PREPARED = None
required = ("runtime_dir", "budget_module", "DEADLINE_MONOTONIC", "time", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("실행 코드 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
if (globals().get("RUNTIME_PREPARED") != (str(runtime_dir), DEADLINE_MONOTONIC)
        or not (runtime_dir / "requirements.txt").is_file()):
    raise RuntimeError("현재 실행의 코드 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")
try:
    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    package_deadline = (None if DEADLINE_MONOTONIC is None else
                        min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60))
    budget_module.run_supervised(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],
        deadline_monotonic=package_deadline, shutdown_margin_s=10,
    )
    DEPENDENCIES_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    if DEADLINE_MONOTONIC is None:
        print("패키지 준비 완료 · 작업 시간 제한 없음")
    else:
        print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")
except BaseException as setup_error:
    reporter = globals().get("save_setup_failure")
    if callable(reporter):
        reporter(setup_error, "dependency_install")
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise RuntimeError("패키지 준비 실패: " + type(setup_error).__name__ + ". Drive 오류 보고서를 확인하세요.") from None


In [ ]:
# @title 실행 모드에 따라 학습 → 결과 저장 → 종료 · GPU 반환
import csv
import json
import math
import subprocess
required = ("runtime_dir", "SETTINGS", "budget_module", "DEADLINE_MONOTONIC",
            "BUDGET_STARTED_MONOTONIC", "time", "Path", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("준비 셀이 완료되지 않았거나 런타임이 초기화됐습니다. 마지막 셀만 실행하지 말고 런타임 → 모두 실행으로 처음부터 실행하세요. 앞선 셀이 실패했다면 그 셀의 첫 오류를 확인하세요.")
prepared_context = (str(runtime_dir), DEADLINE_MONOTONIC)
if (globals().get("RUNTIME_PREPARED") != prepared_context
        or globals().get("DEPENDENCIES_PREPARED") != prepared_context
        or not (runtime_dir / "run_notebook.py").is_file()):
    raise RuntimeError("현재 실행의 코드 또는 패키지 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")
error_file = runtime_dir / "notebook_error.json"
try:
    error_file.write_text(json.dumps({"status": "starting"}), encoding="utf-8")
    if DEADLINE_MONOTONIC is not None and time.monotonic() >= DEADLINE_MONOTONIC:
        raise budget_module.BudgetExceeded("notebook launch", deadline_monotonic=DEADLINE_MONOTONIC)
    settings_file = runtime_dir / "notebook_settings.json"
    result_file = runtime_dir / "notebook_result.json"
    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    budget_module.run_supervised(
        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
         "--settings", str(settings_file), "--result_file", str(result_file)],
        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,
    )
    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))
    print("결과 폴더:", result_dir)
    summary = result_dir / "candidate_summary.csv"
    if summary.is_file():
        print("동일 Training development 분할 비교:")
        with summary.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                      "오심 F1 =", round(float(row["nausea_f1"]), 6),
                      "선택 epoch =", row.get("selected_epochs", "기록 확인"))
    for filename in ["training_metrics.json", "validation_metrics.json"]:
        path = result_dir / filename
        if path.is_file():
            report = json.loads(path.read_text(encoding="utf-8-sig"))
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
            for symptom, values in report["per_symptom"].items():
                print(symptom, "F1 =", round(values["f1"], 4),
                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))
    manifest_path = result_dir / "mission3_ver25_model.ckpt"
    manifest = {}
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        selection = manifest.get("selection", {})
        print("저장된 후보:", selection.get("selected_candidate", "모델 정보 확인"))
        print("저장 모델 학습 범위:", manifest.get("training_scope", "모델 정보 확인"),
              "· 학습 표본:", manifest.get("fitted_samples", "모델 정보 확인"),
              "· 학습 epoch:", selection.get("selected_epochs", "모델 정보 확인"))
    status_path = result_dir / "runtime.json"
    status = json.loads(status_path.read_text(encoding="utf-8-sig")) if status_path.is_file() else {}
    print("실행 상태:", status.get("status", "기록 확인 필요"))
    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")
    print("선택 근거:", result_dir / "final_selection.json")
    components = manifest.get("components", [])
    components_present = bool(components) and all(
        isinstance(item, dict) and isinstance(item.get("model_directory"), str)
        and (result_dir / item["model_directory"]).is_dir() for item in components)
    complete = ((result_dir / "run_complete.json").is_file() and manifest_path.is_file()
                and manifest.get("training_scope") == "all_training" and components_present
                and ((result_dir / "mission3.csv").is_file() or not SETTINGS["run_validation"])
                and status.get("status") in {"complete", "complete_not_improved"})
    pilot_path = result_dir / "pilot_complete.json"
    pilot = json.loads(pilot_path.read_text(encoding="utf-8-sig")) if pilot_path.is_file() else {}
    pilot_complete = (SETTINGS["run_mode"] == "pilot" and status.get("status") == "pilot_complete"
                      and pilot.get("status") == "pilot_complete"
                      and pilot.get("training_scope") == "training_subset"
                      and pilot.get("submission_ready") is False
                      and all((result_dir / name).is_file() for name in
                              ("candidate_summary.csv", "final_selection.json", "full_run_estimate.json"))
                      and not manifest_path.exists() and not (result_dir / "run_complete.json").exists())
    if pilot_complete:
        fitted_samples = pilot.get("fitted_samples", SETTINGS["pilot_samples"])
        print("예비 학습 완료 ·", fitted_samples, "개 학습 결과를 저장하고 종료했습니다.")
        print("전체 Training 학습과 confirmation·공식 Validation 평가는 실행하지 않았습니다.")
        print("후속 전체 학습 예상 시간:", result_dir / "full_run_estimate.json")
        estimate = json.loads((result_dir / "full_run_estimate.json").read_text(encoding="utf-8-sig"))
        recommended_minutes = estimate.get("recommended_budget_minutes") if isinstance(estimate, dict) else None
        if (isinstance(recommended_minutes, (int, float)) and not isinstance(recommended_minutes, bool)
                and math.isfinite(recommended_minutes) and recommended_minutes > 0):
            print("예상 권장 전체 실행 예산:", recommended_minutes, "분 (약",
                  round(recommended_minutes / 60, 1), "시간) · 실측 기반 추정이며 보장 시간은 아닙니다.")
        print("결과를 검토한 뒤 SETTINGS의 run_mode='full', source_run=위 결과 폴더,")
        print("time_budget_minutes=None(시간 제한 없음) 또는 예상 시간을 반영한 예산으로 설정하고 처음부터 모두 실행하세요.")
        print("예비 학습 결과는 최종 제출 모델이 아닙니다.")
    elif complete and SETTINGS["run_mode"] == "full":
        if SETTINGS["run_validation"]:
            print("최종 작업 완료 · 전체 Training 재학습과 저장 검증을 마친 예측:", result_dir / "mission3.csv")
        else:
            print("모델 검증 실행 완료 · Validation 예측과 제출 CSV는 만들지 않았습니다.")
            print("제출용 실행은 run_validation=True로 설정해야 합니다.")
        print("이전 최고 모델과의 성능 비교는 validation_metrics.json으로 따로 확인하세요.")
        if status.get("status") == "complete_not_improved":
            print("새 표현 후보가 대조군보다 개선되지 않음 · 기존 baseline보다 개선한 결과로 표시하지 않습니다.")
    else:
        print("미완료 실행:", status.get("status", "상태 기록 없음"),
              "· 최종 제출 결과로 사용하지 마세요. 결과 폴더의 로그를 확인하세요.")
except Exception as worker_error:
    failure = {"status": "failed", "stage": "notebook_run", "error_type": type(worker_error).__name__,
               "category": "notebook_execution_failed",
               "reason": "실행 또는 결과 확인 중 오류가 발생했습니다. 실패 단계와 코드 위치를 확인하세요.", "frames": []}
    if isinstance(worker_error, subprocess.CalledProcessError):
        failure.update(stage="worker_process", category="worker_process_failed", exit_code=worker_error.returncode,
                       reason="실행 프로그램이 상세 오류를 저장하기 전에 종료됐습니다. 종료 코드와 준비 상태 확인이 필요합니다.")
        try:
            recorded = json.loads(error_file.read_text(encoding="utf-8"))
            if isinstance(recorded, dict) and recorded.get("status") == "failed":
                failure.update(recorded)
        except (OSError, ValueError):
            pass
    elif type(worker_error).__name__ == "BudgetExceeded":
        failure.update(category="time_budget_exhausted",
                       reason="설정한 작업 예산이 끝나 실행을 중단했습니다. 완료 표시가 없는 결과는 제출에 사용하지 마세요.")
    print("ver25 내부 오류 · 실패 단계:", failure["stage"], "· 오류 종류:", failure["error_type"], flush=True)
    print("확인 내용:", failure["reason"], flush=True)
    if failure.get("module_name"):
        print("불러오지 못한 모듈:", failure["module_name"], flush=True)
    for frame in failure.get("frames", [])[-6:]:
        print("코드 위치:", frame["file"], "·", frame["function"], "·", frame["line"], flush=True)
    saved_error = None
    try:
        from datetime import datetime, timezone
        import uuid
        error_dir = Path(SETTINGS["output_dir"]) / "_diagnostics"
        error_dir.mkdir(parents=True, exist_ok=True)
        saved_error = error_dir / ("ver25_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                                  + "_" + uuid.uuid4().hex[:8] + ".json")
        saved_error.write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8")
        print("오류 보고서:", saved_error, flush=True)
    except OSError as save_error:
        saved_error = None
        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, "· 위의 내부 오류 내용을 복사하세요.", flush=True)
    raise RuntimeError("ver25 실행 실패 [" + failure["stage"] + " / " + failure["error_type"] + "]: "
                       + failure["reason"] + (" 오류 보고서: " + str(saved_error) if saved_error else "")) from None
finally:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· Colab에서 직접 연결을 해제하세요.")


## 결과와 완료 상태

결과는 `DCC_AIdeas/results/mission3/ver25/실행시각_고유번호/`에 저장합니다.
- `model_specs.json`, split/data digests: 정확한 모델 revision과 학습 데이터 출처
- `candidate_summary.csv`, `final_selection.json`: development 선택과 고정 recipe
- development 예측 및 지표: 9개 F1/AP, 오심 지표, 공통 오답 변화와 불확실성
- 배치 속도 측정 기록: 후보별 처리량·GPU 메모리·선택 결과
- `runtime.json`: 실행 상태, 실측 시간, 예산 판단
- `pilot_complete.json`: 예비 학습 저장 검증 완료, `submission_ready=false`
- `full_run_estimate.json`: 이후 전체 학습에 필요한 시간의 실측 기반 추정

**기본 pilot는 결과 저장 후 종료합니다. confirmation·공식 Validation·전체 Training refit은 실행하지 않습니다.**
작은 subset의 development 점수는 확장 여부를 판단하는 자료이며 전체 학습의 최종 성능이 아닙니다.
예비 체크포인트와 고정 모델 revision을 포함하므로 실행 폴더 전체를 보관하세요.

## 결과를 보고 전체 학습을 따로 실행하기

1. 예비 결과의 F1·오심 지표와 `full_run_estimate.json`을 확인합니다.
2. 위의 **실행 설정 셀 하나에서** `run_mode`를 `'full'`로 바꿉니다.
3. `source_run`에는 `pilot_complete.json`이 있는 완료된 예비 실행 폴더의 절대 경로를 넣습니다.
4. 시간 제한 없이 끝까지 실행하려면 `time_budget_minutes=None`으로 설정합니다. 시간 제한을 원하면 예상 시간과 준비·저장 여유를 반영해 분 단위 정수로 지정합니다. 기본값은 180분입니다.
5. GPU 런타임에서 **처음부터 모두 실행**합니다. 원래 예비 결과는 보존하고 별도 결과 폴더를 만듭니다.

full 실행은 원본 데이터·분할·고정 학습 조건과 완료된 예비 실행의 무결성을 확인합니다.
같은 모델 revision과 동결한 후보·epoch를 사용하고 현재 GPU의 배치를 다시 측정합니다.
confirmation은 동결한 선택을 한 번 확인하는 용도이며 후보를 다시 고르지 않습니다.
작은 subset에서 이어서 학습하는 대신 선택 recipe만 공개 초기 가중치에서 전체 Training으로 새로 학습합니다.
공식 Validation은 전체 refit 후에만 `run_validation=True`일 때 평가합니다.

full 실행에서만 `mission3_ver25_model.ckpt`, 최종 모델 폴더, `submission/`을 만듭니다.
`validation_metrics.json`, `mission3.csv`는 공식 Validation 평가를 켠 full 실행 결과입니다.
`run_complete.json`은 전체 Training refit과 최종 저장·오프라인 재로딩 검증을 통과했을 때만 만듭니다.

**fit-only / 미완료 결과는 전체 Training 최종 제출 모델이 아닙니다.**
`complete_not_improved`는 새 표현 후보가 대조군보다 개선되지 않은 실행의 전체 Training refit과 저장 검증을 마친 상태입니다.
기존 친구 baseline보다 개선했다는 뜻이 아닙니다. 최종 완료 표시는 전체 refit과 저장 검증이 끝났을 때만 만듭니다.
manifest 파일 하나만 보관하지 말고 실행 폴더 전체를 보관하세요.
NF4 Qwen 및 Qwen 포함 앙상블은 저장된 로컬 가중치로 추론할 때도 CUDA GPU가 필요합니다.
KLUE 단독 모델은 CPU 추론도 지원합니다.
실패 시 원문이나 개별 데이터 파일명을 출력하지 않고 실패 단계·오류 종류·코드 위치를
`ver25/_diagnostics/`에 저장한 뒤 GPU 반환을 요청합니다.
중단된 학습의 자동 재개는 지원하지 않습니다. 자동 반환 실패 시 Colab의 연결 해제/런타임 삭제를 직접 실행하세요.

Qwen은 Apache-2.0, KLUE는 CC-BY-SA-4.0이며 출처와 조건을 패키지에 보존합니다.
의존성은 ver25 전용 `requirements.txt`를 사용합니다. 노트북 의존성 설치는 Colab에서 실행합니다.
